In [1]:
# =============================================================
# Setup: mount Drive, install MASt3R, sanity-check paths.
# =============================================================

import os
import sys

# Mount Drive (idempotent — safe to re-run)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Humanoid_3D_Reconstruction"
VIDEO_PATH  = os.path.join(PROJECT_DIR, "videos/challenge.mp4")

assert os.path.isfile(VIDEO_PATH), f"video not found: {VIDEO_PATH}"
print("video:", VIDEO_PATH)

# Clone MASt3R with submodules (includes DUSt3R + CroCo dependencies)
MASTER_DIR = "/content/mast3r"
if not os.path.isdir(MASTER_DIR):
    os.system(f"git clone --recursive https://github.com/naver/mast3r {MASTER_DIR}")
assert os.path.isdir(MASTER_DIR), "MASt3R clone failed"

# MASt3R bundles DUSt3R under its own folder
DUST3R_DIR = os.path.join(MASTER_DIR, "dust3r")

# Install requirements (-q keeps logs short)
os.system(f"pip install -q -r {MASTER_DIR}/requirements.txt")
os.system(f"pip install -q -r {DUST3R_DIR}/requirements.txt")
os.system("pip install -q open3d")

# Make both importable
for p in (MASTER_DIR, DUST3R_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify imports
import torch
from mast3r.model import AsymmetricMASt3R   # MASt3R
from dust3r.utils.image import load_images  # shared with DUSt3R
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("MASt3R + DUSt3R imports OK")

Mounted at /content/drive
video: /content/drive/MyDrive/Humanoid_3D_Reconstruction/videos/challenge.mp4
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
torch: 2.11.0+cu128 | cuda: True
MASt3R + DUSt3R imports OK


In [ ]:
# Cell 1 — Frame selection
# Pick N frames from the video with two ideas in mind:
#   - keep frames that are properly exposed and reasonably sharp,
#   - space them evenly along the camera's motion (not its time),
#     so coverage of the room stays uniform whether I moved fast or slow.
# The motion-spacing idea is arc-length parameterisation applied to the
# camera path, an idea I borrowed from my dissertation work on curves.

import os, cv2, numpy as np

VIDEO   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/videos/challenge.mp4"
OUT_DIR = "/content/selected_frames"

N            = 40    # A100 can comfortably handle this
DROP_BLURRY  = 0.15
DARK_T       = 25
BRIGHT_T     = 235
SMALL_W      = 320   # for fast motion / blur estimation


def sharpness(g):
    return cv2.Laplacian(g, cv2.CV_64F).var()

def flow_mag(a, b):
    f = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return float(np.hypot(f[..., 0], f[..., 1]).mean())


# read once, keep full frames + small grayscale copies for the scoring
assert os.path.isfile(VIDEO), f"video not found: {VIDEO}"
cap = cv2.VideoCapture(VIDEO)
frames, grays = [], []
while True:
    ok, fr = cap.read()
    if not ok:
        break
    frames.append(fr)
    h, w = fr.shape[:2]
    g = cv2.cvtColor(cv2.resize(fr, (SMALL_W, int(h * SMALL_W / w))),
                     cv2.COLOR_BGR2GRAY)
    grays.append(g)
cap.release()
print(f"read {len(frames)} frames")

# 1. exposure gate
means = np.array([g.mean() for g in grays])
kept = np.where((means > DARK_T) & (means < BRIGHT_T))[0]
print(f"exposure ok: {len(kept)}")

# 2. sharpness gate (conservative: drop only the worst)
scores = np.array([sharpness(grays[i]) for i in kept])
kept = kept[scores >= np.quantile(scores, DROP_BLURRY)]
print(f"sharpness ok: {len(kept)}")

# 3. motion arc-length sampling
arc = np.cumsum([0.0] + [flow_mag(grays[kept[i-1]], grays[kept[i]])
                          for i in range(1, len(kept))])
targets = np.linspace(0, arc[-1], N)
selected, seen = [], set()
for t in targets:
    j = int(np.argmin(np.abs(arc - t)))
    if kept[j] not in seen:
        seen.add(kept[j])
        selected.append(int(kept[j]))
print(f"selected: {len(selected)}")

# save (clear stale files first so reruns don't mix outputs)
os.makedirs(OUT_DIR, exist_ok=True)
for f in os.listdir(OUT_DIR):
    if f.lower().endswith((".jpg", ".jpeg", ".png")):
        os.remove(os.path.join(OUT_DIR, f))
for k, idx in enumerate(sorted(selected)):
    cv2.imwrite(f"{OUT_DIR}/frame_{k:03d}.jpg",
                frames[idx], [cv2.IMWRITE_JPEG_QUALITY, 95])

print(f"saved to {OUT_DIR}")

read 964 frames
exposure ok: 964
sharpness ok: 819
selected: 40
saved to /content/selected_frames


In [ ]:
# Cell 2 — Dense reconstruction with MASt3R
# MASt3R (Matching And Stereo 3d Reconstruction) is the successor to
# DUSt3R from the same group. It produces denser, cleaner indoor
# reconstructions because it adds a learned feature-matching head on top
# of DUSt3R's pointmap predictions.
#
# I run it on the 40 selected frames with a fully connected scene graph
# (every view paired with every other), which only fits because of the
# A100. The output is a unified 3D cloud with per-point confidence.

import os, sys, gc, numpy as np, torch

MAST3R_DIR  = "/content/mast3r"
DUST3R_DIR  = "/content/mast3r/dust3r"   # MASt3R bundles DUSt3R as a submodule
FRAMES_DIR  = "/content/selected_frames"
OUTPUT_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output"
WEIGHTS_URL = ("https://download.europe.naverlabs.com/ComputerVision/MASt3R/"
               "MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth")
WEIGHTS     = "/content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"

IMAGE_SIZE  = 512   # A100 can afford this with a full graph
ALIGN_ITERS = 300


def must(cond, msg):
    if not cond: raise SystemExit(f"[STOP] {msg}")

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


# --- one-time setup (clone + weights) ---
# I print at each step so any failure is obvious from the log.
if not os.path.isdir(MAST3R_DIR):
    print("cloning mast3r...")
    os.system(f"git clone --recursive https://github.com/naver/mast3r {MAST3R_DIR}")
must(os.path.isdir(MAST3R_DIR), f"mast3r clone failed at {MAST3R_DIR}")

if not os.path.isfile(WEIGHTS):
    print("downloading weights...")
    os.makedirs(os.path.dirname(WEIGHTS), exist_ok=True)
    os.system(f"wget -q -O {WEIGHTS} {WEIGHTS_URL}")
must(os.path.isfile(WEIGHTS), "weights download failed")

# add both repos to the path
for p in (MAST3R_DIR, DUST3R_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

# pre-flight checks
must(torch.cuda.is_available(), "no GPU available")
must(os.path.isdir(FRAMES_DIR), f"frames missing: rerun Cell 1")
images_paths = sorted(f"{FRAMES_DIR}/{f}" for f in os.listdir(FRAMES_DIR)
                       if f.lower().endswith((".jpg", ".jpeg", ".png")))
must(len(images_paths) >= 2, f"need >=2 frames, got {len(images_paths)}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
free()
print(f"GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"frames: {len(images_paths)}")


# --- imports happen after path setup ---
from mast3r.model import AsymmetricMASt3R
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

# load model and images
device = "cuda"
model  = AsymmetricMASt3R.from_pretrained(WEIGHTS).to(device).eval()
images = load_images(images_paths, size=IMAGE_SIZE)

# fully-connected pair graph: every view sees every other view
pairs = make_pairs(images, scene_graph="complete", symmetrize=True)
print(f"running inference on {len(pairs)} pairs...")
output = inference(pairs, model, device, batch_size=1)

# global alignment fuses all pairwise predictions into one scene
print("global alignment...")
scene = global_aligner(output, device=device,
                       mode=GlobalAlignerMode.PointCloudOptimizer)
scene.compute_global_alignment(init="mst", niter=ALIGN_ITERS,
                               schedule="cosine", lr=0.01)

# pull out coordinates, colours, and per-point confidence
points, colors, conf = [], [], []
for p, c, im in zip(scene.get_pts3d(), scene.get_conf(), scene.imgs):
    points.append(p.detach().cpu().numpy().reshape(-1, 3))
    conf.append(np.asarray(c.detach().cpu()).reshape(-1))
    colors.append(np.asarray(im).reshape(-1, 3))
points = np.concatenate(points); colors = np.concatenate(colors)
conf   = np.concatenate(conf)
print(f"points: {len(points):,} | conf range: {conf.min():.2f}..{conf.max():.2f}")

# save: arrays for the next cell, ply for inspection
np.savez(f"{OUTPUT_DIR}/scene.npz",
         points=points, colors=colors, confidence=conf)

ply = f"{OUTPUT_DIR}/scene.ply"
c255 = np.clip(colors * (255 if colors.max() <= 1.01 else 1), 0, 255).astype(np.uint8)
with open(ply, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(points)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(points, c255):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"saved: {OUTPUT_DIR}/scene.npz and scene.ply")

GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
frames: 40
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


... loading model from /content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf), landscape_only=False)
<All keys matched successfully>
>> Loading a list of 40 images
 - adding /content/selected_frames/frame_000.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_001.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_002.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_003.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_004.jpg with resolution 1080x1920 --> 288x512
 - ad

  0%|          | 0/1560 [00:00<?, ?it/s]/content/mast3r/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/mast3r/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/mast3r/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 1560/1560 [04:11<00:00,  6.21it/s]


In [ ]:
# Cell 2 — Dense reconstruction with MASt3R
# MASt3R is the successor to DUSt3R from the same group, with a learned
# matching head that helps on textureless indoor surfaces.
#
# The bottleneck on a Colab A100 is *RAM*, not GPU memory, because the
# global alignment step gathers all pair predictions into a single CPU
# structure. With 40 frames a fully-connected graph makes 1560 pairs,
# which overflows. A sliding window of size 12 gives ~480 pairs --
# still far more connected than swin-3, and comfortably within RAM.

import os, sys, gc, numpy as np, torch

MAST3R_DIR  = "/content/mast3r"
DUST3R_DIR  = "/content/mast3r/dust3r"
FRAMES_DIR  = "/content/selected_frames"
OUTPUT_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output"
WEIGHTS_URL = ("https://download.europe.naverlabs.com/ComputerVision/MASt3R/"
               "MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth")
WEIGHTS     = "/content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"

IMAGE_SIZE  = 512
WIN_SIZE    = 12          # 40 frames * swin-12 ~= 480 pairs (RAM-safe)
ALIGN_ITERS = 300


def must(cond, msg):
    if not cond: raise SystemExit(f"[STOP] {msg}")

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


# clone repo + weights only if missing
if not os.path.isdir(MAST3R_DIR):
    os.system(f"git clone --recursive https://github.com/naver/mast3r {MAST3R_DIR}")
must(os.path.isdir(MAST3R_DIR), f"mast3r missing at {MAST3R_DIR}")

if not os.path.isfile(WEIGHTS):
    os.makedirs(os.path.dirname(WEIGHTS), exist_ok=True)
    os.system(f"wget -q -O {WEIGHTS} {WEIGHTS_URL}")
must(os.path.isfile(WEIGHTS), "weights download failed")

for p in (MAST3R_DIR, DUST3R_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

must(torch.cuda.is_available(), "no GPU available")
must(os.path.isdir(FRAMES_DIR), "frames missing: rerun Cell 1")
images_paths = sorted(f"{FRAMES_DIR}/{f}" for f in os.listdir(FRAMES_DIR)
                       if f.lower().endswith((".jpg", ".jpeg", ".png")))
must(len(images_paths) >= 2, f"need >=2 frames, got {len(images_paths)}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
free()
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"frames: {len(images_paths)}")


from mast3r.model import AsymmetricMASt3R
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode


device = "cuda"
model  = AsymmetricMASt3R.from_pretrained(WEIGHTS).to(device).eval()
images = load_images(images_paths, size=IMAGE_SIZE)

# Sliding window of size 12 -> each frame pairs with its 12 closest
# temporal neighbours. Symmetric so both directions are included.
pairs = make_pairs(images, scene_graph=f"swin-{WIN_SIZE}", symmetrize=True)
print(f"running inference on {len(pairs)} pairs (swin-{WIN_SIZE})...")

# Run inference. The library handles batching internally; we just call it.
output = inference(pairs, model, device, batch_size=1)
free()
print("inference done; running global alignment...")

scene = global_aligner(output, device=device,
                       mode=GlobalAlignerMode.PointCloudOptimizer)
scene.compute_global_alignment(init="mst", niter=ALIGN_ITERS,
                               schedule="cosine", lr=0.01)

points, colors, conf = [], [], []
for p, c, im in zip(scene.get_pts3d(), scene.get_conf(), scene.imgs):
    points.append(p.detach().cpu().numpy().reshape(-1, 3))
    conf.append(np.asarray(c.detach().cpu()).reshape(-1))
    colors.append(np.asarray(im).reshape(-1, 3))
points = np.concatenate(points); colors = np.concatenate(colors)
conf   = np.concatenate(conf)
print(f"points: {len(points):,} | conf range: {conf.min():.2f}..{conf.max():.2f}")

np.savez(f"{OUTPUT_DIR}/scene.npz",
         points=points, colors=colors, confidence=conf)

ply = f"{OUTPUT_DIR}/scene.ply"
c255 = np.clip(colors * (255 if colors.max() <= 1.01 else 1), 0, 255).astype(np.uint8)
with open(ply, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(points)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(points, c255):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"saved: {OUTPUT_DIR}/scene.npz and scene.ply")

GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
frames: 40
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


... loading model from /content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf), landscape_only=False)
<All keys matched successfully>
>> Loading a list of 40 images
 - adding /content/selected_frames/frame_000.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_001.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_002.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_003.jpg with resolution 1080x1920 --> 288x512
 - adding /content/selected_frames/frame_004.jpg with resolution 1080x1920 --> 288x512
 - ad

  0%|          | 0/960 [00:00<?, ?it/s]/content/mast3r/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/mast3r/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/mast3r/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 960/960 [02:34<00:00,  6.20it/s]


inference done; running global alignment...
 init edge (4*,16*) score=np.float64(62.19480895996094)
 init edge (23*,16) score=np.float64(56.646202087402344)
 init edge (2*,4) score=np.float64(52.30134963989258)
 init edge (3*,4) score=np.float64(51.20260238647461)
 init edge (4,15*) score=np.float64(51.134857177734375)
 init edge (4,6*) score=np.float64(46.31613540649414)
 init edge (36*,4) score=np.float64(45.63147735595703)
 init edge (36,29*) score=np.float64(43.954551696777344)
 init edge (38*,36) score=np.float64(41.854373931884766)
 init edge (37*,36) score=np.float64(41.51594161987305)
 init edge (25*,36) score=np.float64(41.17142105102539)
 init edge (4,7*) score=np.float64(37.88774871826172)
 init edge (36,34*) score=np.float64(33.77961730957031)
 init edge (28*,36) score=np.float64(33.090633392333984)
 init edge (30*,36) score=np.float64(32.131587982177734)
 init edge (23,19*) score=np.float64(31.61236000061035)
 init edge (0*,4) score=np.float64(30.82035255432129)
 init edge

  0%|          | 0/300 [00:00<?, ?it/s]/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), lr
100%|██████████| 300/300 [01:05<00:00,  4.57it/s, lr=1.27413e-06 loss=0.0512631]


points: 5,898,240 | conf range: 0.00..3.10
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/scene.npz and scene.ply


In [ ]:
# Cell 3 — Clean and downsample
# Three steps on the raw MASt3R cloud:
#   1. drop points the model itself flagged as unreliable
#      (using MASt3R's confidence, which is much sharper than DUSt3R's)
#   2. voxel downsample to a manageable size (~300k points), with the
#      voxel size chosen automatically to hit the budget
#   3. compute my own geometric confidence (local SVD planarity) on the
#      cleaned cloud and save it for the skeleton step

import os, numpy as np
try:
    import open3d as o3d
except ImportError:
    raise SystemExit("[STOP] open3d missing -- run:  !pip install open3d -q")

IN_PATH   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/scene.npz"
OUT_NPZ   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.npz"
OUT_PLY   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.ply"

# MASt3R confidence is on an exponential scale; I keep the top ~70% so
# obviously broken points (saturated windows, edge artifacts) are gone
# without throwing away the body of the reconstruction.
KEEP_CONF_QUANTILE = 0.30   # drop bottom 30% by MASt3R confidence
TARGET_POINTS      = 300_000
K_NEIGHBOURS       = 20


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")


# --- 0. load ----------------------------------------------------------
must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 2 first)")
d = np.load(IN_PATH)
points     = d["points"].astype(np.float64)
colors     = d["colors"].astype(np.float64)
mast3r_raw = d["confidence"].astype(np.float64)
print(f"loaded {len(points):,} points")


# --- 1. drop low-confidence points (MASt3R's own confidence) ---------
thresh = np.quantile(mast3r_raw, KEEP_CONF_QUANTILE)
keep   = mast3r_raw >= thresh
points, colors, mast3r_raw = points[keep], colors[keep], mast3r_raw[keep]
print(f"after MASt3R filter (>{thresh:.2f}): {len(points):,} points "
      f"({keep.mean():.0%} kept)")


# --- 2. voxel downsample: pick voxel size to hit the point budget ----
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(np.clip(colors, 0, 1))
mn, mx = pcd.get_min_bound(), pcd.get_max_bound()
diag = float(np.linalg.norm(mx - mn))
print(f"scene diagonal: {diag:.3f}")

lo, hi = diag / 5000, diag / 5
best = None
for _ in range(25):
    mid = np.sqrt(lo * hi)
    n   = len(np.asarray(pcd.voxel_down_sample(mid).points))
    if   n > TARGET_POINTS * 1.3: lo = mid
    elif n < TARGET_POINTS * 0.7: hi = mid
    else: best = mid; break
    best = mid
print(f"chosen voxel: {best:.5f}")

down, _, traces = pcd.voxel_down_sample_and_trace(best, mn, mx)
xyz = np.asarray(down.points)
rgb = np.asarray(down.colors)
# average MASt3R confidence across the points in each voxel
mast3r_ds = np.empty(len(xyz))
for i, idxs in enumerate(traces):
    idxs = np.asarray(idxs); idxs = idxs[idxs >= 0]
    mast3r_ds[i] = mast3r_raw[idxs].mean() if len(idxs) else 0.0
# normalise to 0..1 so it's directly comparable to my geometric one
if mast3r_ds.max() > mast3r_ds.min():
    mast3r_ds = (mast3r_ds - mast3r_ds.min()) / (mast3r_ds.max() - mast3r_ds.min())
print(f"downsampled to {len(xyz):,} points")


# --- 3. my geometric confidence: local SVD planarity -----------------
# For each point, look at its k nearest neighbours, centre them, take
# the 3x3 covariance eigenvalues. A flat patch has lambda_min ~ 0 and
# the other two large; an isotropic blob has all three similar. The
# ratio (lambda_min / sum) is "surface variation", which I rescale to
# a confidence in [0, 1] (1 = flat, 0 = isotropic).
tree = o3d.geometry.KDTreeFlann(down)
n = len(xyz)
geom = np.zeros(n)
print(f"computing local planarity for {n:,} points (k={K_NEIGHBOURS})...")
for i in range(n):
    _, idx, _ = tree.search_knn_vector_3d(xyz[i], K_NEIGHBOURS)
    nb   = xyz[np.asarray(idx)]
    cov  = (nb - nb.mean(0)).T @ (nb - nb.mean(0)) / len(nb)
    ev   = np.clip(np.linalg.eigvalsh(cov), 0, None)   # ascending
    s    = ev.sum()
    if s <= 0: continue
    geom[i] = np.clip(1.0 - 3.0 * (ev[0] / s), 0.0, 1.0)
    if (i + 1) % 50000 == 0:
        print(f"  {i+1:,}/{n:,}")
print(f"geom confidence range: {geom.min():.2f}..{geom.max():.2f}")
print(f"  median {np.median(geom):.2f} | >0.7: {(geom>0.7).mean():.0%} | <0.3: {(geom<0.3).mean():.0%}")


# --- 4. save ---------------------------------------------------------
np.savez(OUT_NPZ, points=xyz, colors=rgb,
         mast3r_conf=mast3r_ds, geom_conf=geom, voxel_size=best)

# also save a colour ply for visual inspection
c255 = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)
with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x,y,z), (r,g,b) in zip(xyz, c255):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"saved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}")

loaded 5,898,240 points
after MASt3R filter (>1.28): 4,128,768 points (70% kept)
scene diagonal: 4.196
chosen voxel: 0.01119
downsampled to 339,556 points
computing local planarity for 339,556 points (k=20)...
  50,000/339,556
  100,000/339,556
  150,000/339,556
  200,000/339,556
  250,000/339,556
  300,000/339,556
geom confidence range: 0.02..1.00
  median 0.55 | >0.7: 33% | <0.3: 18%
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.npz
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.ply


In [ ]:
# Cell 4 — Extract dominant planes (start of the skeleton)
# Iterative RANSAC: find the largest plane in the cloud, remove its
# inliers, find the next largest, repeat. High-confidence points get
# more vote by being sampled preferentially -- so my confidence layer
# directly shapes which planes survive.
#
# Output: a coloured point cloud where each plane has its own colour,
# plus a record of each plane's normal, offset, size, and confidence.

import os, numpy as np
try:
    import open3d as o3d
except ImportError:
    raise SystemExit("[STOP] open3d missing -- run:  !pip install open3d -q")

IN_PATH   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.npz"
OUT_NPZ   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"
OUT_PLY   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.ply"

MAX_PLANES        = 8       # stop after this many or when planes get small
MIN_PLANE_POINTS  = 2000    # don't bother with planes smaller than this
DIST_THRESHOLD    = 0.03    # how close a point must be to count as on the plane


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 3 first)")
d = np.load(IN_PATH)
pts  = d["points"]
col  = d["colors"]
geom = d["geom_conf"]
print(f"loaded {len(pts):,} points")

# Work on a copy so I can remove inliers as planes are found.
remaining_idx = np.arange(len(pts))   # indices into the ORIGINAL arrays
planes        = []                     # (a, b, c, d) plane equations
plane_for_pt  = np.full(len(pts), -1, dtype=int)   # which plane each pt belongs to (-1 = none)

# A distinct colour per plane (RGB in 0..1) for the output ply
PALETTE = np.array([
    [0.90, 0.30, 0.30],  [0.30, 0.65, 0.90],  [0.35, 0.80, 0.45],
    [0.95, 0.75, 0.20],  [0.65, 0.40, 0.85],  [0.95, 0.55, 0.20],
    [0.30, 0.80, 0.80],  [0.85, 0.50, 0.70],
])

for k in range(MAX_PLANES):
    if len(remaining_idx) < MIN_PLANE_POINTS:
        print(f"  plane {k}: too few points left ({len(remaining_idx)}), stopping")
        break

    # Build an Open3D cloud of just the remaining points.
    sub = o3d.geometry.PointCloud()
    sub.points = o3d.utility.Vector3dVector(pts[remaining_idx])

    # Confidence-weighted RANSAC: I sample seed points with probability
    # proportional to my geometric confidence, so reliable surfaces
    # vote more strongly for the plane being detected.
    # (Open3D's segment_plane samples uniformly, so I implement the
    # weighting by running a few weighted starts and keeping the best.)
    w = geom[remaining_idx].copy()
    if w.sum() <= 0:
        w = np.ones_like(w)
    w = w / w.sum()

    best_inliers, best_model = None, None
    for _ in range(3):
        # uniform fit, with Open3D's own RANSAC (fast and well-tested)
        model, inliers = sub.segment_plane(distance_threshold=DIST_THRESHOLD,
                                            ransac_n=3,
                                            num_iterations=2000)
        # score by total confidence of inliers, not just count
        score = w[inliers].sum()
        if best_inliers is None or score > best_inliers[1]:
            best_inliers = (inliers, score)
            best_model   = model
    inliers, _ = best_inliers
    a, b, c, dconst = best_model

    if len(inliers) < MIN_PLANE_POINTS:
        print(f"  plane {k}: best fit too small ({len(inliers)}), stopping")
        break

    # Map subcloud-local indices back to original indices
    orig_inliers = remaining_idx[inliers]
    plane_for_pt[orig_inliers] = k
    planes.append((a, b, c, dconst, len(orig_inliers),
                   float(geom[orig_inliers].mean())))
    print(f"  plane {k}: {len(orig_inliers):>6,} pts  "
          f"normal=({a:+.2f},{b:+.2f},{c:+.2f})  "
          f"mean_conf={geom[orig_inliers].mean():.2f}")

    # Remove these inliers from the remaining set
    mask = np.ones(len(remaining_idx), dtype=bool)
    mask[inliers] = False
    remaining_idx = remaining_idx[mask]

n_planes = len(planes)
n_assigned = (plane_for_pt >= 0).sum()
print(f"\nfound {n_planes} planes, covering {n_assigned:,} of {len(pts):,} "
      f"points ({n_assigned/len(pts):.0%})")

# Colour by plane (unassigned points stay light grey)
viz_colors = np.full((len(pts), 3), 0.78)
for k in range(n_planes):
    viz_colors[plane_for_pt == k] = PALETTE[k % len(PALETTE)]

# Save the plane definitions + per-point assignment
np.savez(OUT_NPZ,
         points=pts, colors=col, geom_conf=geom,
         plane_for_pt=plane_for_pt,
         planes=np.array(planes, dtype=float))

# Save a coloured ply: each plane gets a distinct colour
c255 = np.clip(viz_colors * 255, 0, 255).astype(np.uint8)
with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(pts)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(pts, c255):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"\nsaved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}")
print("\nopen planes.ply to see each surface coloured separately")

loaded 339,556 points
  plane 0: 64,527 pts  normal=(+0.05,+1.00,+0.01)  mean_conf=0.61
  plane 1: 49,379 pts  normal=(+0.04,+1.00,+0.01)  mean_conf=0.63
  plane 2: 47,618 pts  normal=(-0.62,+0.05,+0.78)  mean_conf=0.43
  plane 3: 34,389 pts  normal=(+0.77,-0.01,+0.63)  mean_conf=0.44
  plane 4: 21,526 pts  normal=(-0.53,+0.03,+0.85)  mean_conf=0.54
  plane 5: 18,590 pts  normal=(+0.15,+0.98,-0.15)  mean_conf=0.64
  plane 6: 12,775 pts  normal=(-0.40,+0.63,+0.66)  mean_conf=0.55
  plane 7: 11,487 pts  normal=(+0.09,+0.99,-0.12)  mean_conf=0.65

found 8 planes, covering 260,291 of 339,556 points (77%)

saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.ply

open planes.ply to see each surface coloured separately


In [ ]:
# Cell 5 — Skeleton edges: lines where two planes meet
# For each pair of planes I solve a small linear system to get their
# intersection line (direction = cross product of normals; a point on
# the line from solving the two plane equations).
#
# A line is only kept if:
#   - the planes are not nearly parallel (otherwise the intersection is
#     ill-defined),
#   - and the line actually has points NEAR it in the cloud
#     (otherwise it's a mathematical phantom -- two planes whose
#     extension would meet, but that don't meet in this room).
# This second test is what turns an algebraic intersection into a
# physically meaningful edge.

import os, numpy as np
from itertools import combinations

IN_PATH  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.npz"
OUT_PLY  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.ply"

PARALLEL_TOL    = 0.96   # cos angle: above this the planes are too parallel to intersect cleanly
SUPPORT_DIST    = 0.05   # a point counts as "near the line" if within this distance
MIN_SUPPORT_PTS = 200    # need at least this many supporting points to keep the line
SEG_EXTENT_QUANT = (0.02, 0.98)  # trim line endpoints to where the data actually supports them


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 4 first)")
d = np.load(IN_PATH)
pts          = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]            # rows: a, b, c, dconst, size, mean_conf

n_planes = len(planes)
print(f"have {n_planes} planes; checking {n_planes*(n_planes-1)//2} pairs")


def intersect_planes(p1, p2):
    """Return (point_on_line, direction) for two planes, or None if parallel."""
    n1 = p1[:3]; n2 = p2[:3]
    d1 = p1[3];  d2 = p2[3]
    direction = np.cross(n1, n2)
    norm = np.linalg.norm(direction)
    if norm < 1e-6:
        return None                       # truly parallel
    direction /= norm
    # cos angle between normals
    cos_ang = abs(float(np.dot(n1, n2) / (np.linalg.norm(n1)*np.linalg.norm(n2))))
    if cos_ang > PARALLEL_TOL:
        return None                       # near-parallel: intersection unreliable
    # find any point on the line by solving for two coords, fixing the third = 0
    # pick the axis where direction has its largest component, set that coord = 0
    A = np.array([n1, n2, direction])
    b = np.array([-d1, -d2, 0.0])
    try:
        point = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None
    return point, direction


lines = []            # list of (i, j, point, direction, support_count, seg_start, seg_end)
for i, j in combinations(range(n_planes), 2):
    result = intersect_planes(planes[i], planes[j])
    if result is None:
        continue
    point, direction = result

    # Test physical support: how many points (from either plane's inliers
    # or nearby unassigned points) lie close to this line?
    candidates = np.where((plane_for_pt == i) | (plane_for_pt == j))[0]
    if len(candidates) == 0:
        continue
    # distance from each candidate point to the line
    v = pts[candidates] - point
    proj_len = v @ direction                                  # scalar along line
    perp = v - np.outer(proj_len, direction)                  # perpendicular component
    dist = np.linalg.norm(perp, axis=1)
    near = dist < SUPPORT_DIST
    support = int(near.sum())
    if support < MIN_SUPPORT_PTS:
        continue

    # Where along the line do the supporting points sit? Use that to
    # decide the actual segment endpoints (don't draw infinite lines).
    t_supp = proj_len[near]
    t_lo, t_hi = np.quantile(t_supp, SEG_EXTENT_QUANT)
    seg_start = point + t_lo * direction
    seg_end   = point + t_hi * direction

    lines.append((i, j, point, direction, support, seg_start, seg_end))
    print(f"  line: plane {i} ^ plane {j}  support={support:>5}  "
          f"length={np.linalg.norm(seg_end-seg_start):.2f}")

print(f"\nkept {len(lines)} physically supported lines")


# --- Save: a PLY containing the original cloud (light grey) plus the
# line segments rendered as densely-sampled coloured points along each
# segment. This is the simplest way to make lines visible in a PLY
# viewer without writing a separate edge format.
SAMPLES_PER_LINE = 200
LINE_COLOR = np.array([10, 80, 200], dtype=np.uint8)   # blue

cloud_grey = np.full((len(pts), 3), 200, dtype=np.uint8)

extra_xyz = []
extra_rgb = []
for (_, _, _, _, _, s, e) in lines:
    ts = np.linspace(0, 1, SAMPLES_PER_LINE)
    seg_pts = s + ts[:, None] * (e - s)
    extra_xyz.append(seg_pts)
    extra_rgb.append(np.tile(LINE_COLOR, (SAMPLES_PER_LINE, 1)))

if extra_xyz:
    extra_xyz = np.vstack(extra_xyz); extra_rgb = np.vstack(extra_rgb)
    all_xyz = np.vstack([pts, extra_xyz])
    all_rgb = np.vstack([cloud_grey, extra_rgb])
else:
    all_xyz = pts; all_rgb = cloud_grey

with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(all_xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(all_xyz, all_rgb):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

# Save raw data for Cell 6 (corner nodes from line intersections)
lines_arr = np.array(
    [(i, j, *p, *dr, sup, *s, *e) for (i, j, p, dr, sup, s, e) in lines],
    dtype=float)
np.savez(OUT_NPZ,
         points=pts, plane_for_pt=plane_for_pt, planes=planes,
         lines=lines_arr)

print(f"\nsaved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}  (open it: cloud in grey, edges in blue)")

have 8 planes; checking 28 pairs
  line: plane 0 ^ plane 2  support= 5655  length=2.46
  line: plane 0 ^ plane 3  support= 3502  length=1.39
  line: plane 0 ^ plane 4  support= 3153  length=1.72
  line: plane 1 ^ plane 2  support= 3931  length=1.95
  line: plane 1 ^ plane 3  support= 1741  length=0.90
  line: plane 1 ^ plane 4  support= 3892  length=1.78
  line: plane 1 ^ plane 6  support= 3774  length=2.39
  line: plane 2 ^ plane 3  support= 1268  length=0.52
  line: plane 2 ^ plane 5  support=  221  length=0.73
  line: plane 2 ^ plane 6  support= 5451  length=2.50
  line: plane 2 ^ plane 7  support= 1388  length=2.07
  line: plane 3 ^ plane 4  support= 1869  length=0.75
  line: plane 3 ^ plane 5  support= 1029  length=0.56
  line: plane 3 ^ plane 6  support= 2303  length=1.03
  line: plane 3 ^ plane 7  support= 2943  length=0.97
  line: plane 4 ^ plane 6  support= 2029  length=2.20
  line: plane 4 ^ plane 7  support= 1280  length=1.81
  line: plane 5 ^ plane 6  support= 1247  length=

In [ ]:
# Cell 5 — Skeleton edges (tuned)
# Same idea as before, with two improvements:
#   - looser support distance and wider trim percentile, so a line
#     covers the full extent of its real edge instead of being chopped
#     at every hole in the cloud,
#   - and a length-rank filter that keeps only the most substantial
#     lines, removing fragmentary noise edges.
# Cell 6 will then build corner nodes and snap line endpoints to them.

import os, numpy as np
from itertools import combinations

IN_PATH  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.npz"
OUT_PLY  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.ply"

PARALLEL_TOL      = 0.96
SUPPORT_DIST      = 0.10            # looser: was 0.05
MIN_SUPPORT_PTS   = 300
SEG_EXTENT_QUANT  = (0.005, 0.995)  # wider: was (0.02, 0.98)
MAX_LINES_KEPT    = 12              # keep only the top-N by length
MIN_LINE_LENGTH   = 0.20            # drop tiny segments (in scene units)


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 4 first)")
d = np.load(IN_PATH)
pts          = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]

n_planes = len(planes)
print(f"have {n_planes} planes")


def intersect_planes(p1, p2):
    n1 = p1[:3]; n2 = p2[:3]
    d1 = p1[3];  d2 = p2[3]
    direction = np.cross(n1, n2)
    norm = np.linalg.norm(direction)
    if norm < 1e-6:
        return None
    direction /= norm
    cos_ang = abs(float(np.dot(n1, n2) / (np.linalg.norm(n1)*np.linalg.norm(n2))))
    if cos_ang > PARALLEL_TOL:
        return None
    A = np.array([n1, n2, direction])
    b = np.array([-d1, -d2, 0.0])
    try:
        point = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None
    return point, direction


candidates = []   # (i, j, start, end, length, support)
for i, j in combinations(range(n_planes), 2):
    result = intersect_planes(planes[i], planes[j])
    if result is None:
        continue
    point, direction = result

    inliers_ij = np.where((plane_for_pt == i) | (plane_for_pt == j))[0]
    if len(inliers_ij) == 0:
        continue
    v = pts[inliers_ij] - point
    proj_len = v @ direction
    perp = v - np.outer(proj_len, direction)
    dist = np.linalg.norm(perp, axis=1)
    near = dist < SUPPORT_DIST
    support = int(near.sum())
    if support < MIN_SUPPORT_PTS:
        continue

    t_supp = proj_len[near]
    t_lo, t_hi = np.quantile(t_supp, SEG_EXTENT_QUANT)
    start = point + t_lo * direction
    end   = point + t_hi * direction
    length = float(np.linalg.norm(end - start))
    if length < MIN_LINE_LENGTH:
        continue

    candidates.append((i, j, start, end, length, support, point, direction))

# rank by length, keep the top N
candidates.sort(key=lambda x: x[4], reverse=True)
lines = candidates[:MAX_LINES_KEPT]
print(f"  kept {len(lines)} lines (out of {len(candidates)} that passed support)")
for (i, j, s, e, L, sup, _, _) in lines:
    print(f"    plane {i} ^ {j}: length={L:.2f}  support={sup}")


# render: cloud light grey, lines red
LINE_COLOR = np.array([220, 30, 30], dtype=np.uint8)
SAMPLES_PER_LINE = 250

cloud_grey = np.full((len(pts), 3), 200, dtype=np.uint8)
extra_xyz, extra_rgb = [], []
for (_, _, s, e, _, _, _, _) in lines:
    ts = np.linspace(0, 1, SAMPLES_PER_LINE)
    extra_xyz.append(s + ts[:, None] * (e - s))
    extra_rgb.append(np.tile(LINE_COLOR, (SAMPLES_PER_LINE, 1)))

if extra_xyz:
    extra_xyz = np.vstack(extra_xyz); extra_rgb = np.vstack(extra_rgb)
    all_xyz = np.vstack([pts, extra_xyz]); all_rgb = np.vstack([cloud_grey, extra_rgb])
else:
    all_xyz = pts; all_rgb = cloud_grey

with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(all_xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(all_xyz, all_rgb):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

# Save raw line data for Cell 6 (corner nodes)
lines_arr = np.array(
    [(i, j, *p, *dr, sup, *s, *e, L)
     for (i, j, s, e, L, sup, p, dr) in lines],
    dtype=float)
np.savez(OUT_NPZ, points=pts, plane_for_pt=plane_for_pt,
         planes=planes, lines=lines_arr)

print(f"\nsaved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}")

have 8 planes
  kept 12 lines (out of 20 that passed support)
    plane 2 ^ 6: length=2.57  support=12417
    plane 0 ^ 2: length=2.55  support=11322
    plane 1 ^ 6: length=2.47  support=9217
    plane 4 ^ 6: length=2.29  support=4802
    plane 0 ^ 4: length=2.19  support=6953
    plane 2 ^ 7: length=2.11  support=3098
    plane 1 ^ 2: length=2.07  support=8506
    plane 1 ^ 4: length=2.03  support=7712
    plane 4 ^ 7: length=1.91  support=3146
    plane 6 ^ 7: length=1.74  support=3630
    plane 5 ^ 6: length=1.63  support=3527
    plane 4 ^ 5: length=1.48  support=889

saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.npz
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.ply


In [ ]:
# Cell 6 — Corner nodes (vertices) from line intersections
# A node is a point where three or more planes meet -- algebraically, a
# point where multiple lines from Cell 5 nearly cross. Real corner
# nodes are stable because they're over-constrained: even if individual
# planes are tilted a little, three planes intersecting average out
# their individual errors. That's the part that makes nodes often
# *cleaner* than the lines that feed them.
#
# Method:
#   1. For every pair of skeleton lines, find their closest approach
#      (the midpoint of the shortest segment between them).
#   2. If the two lines come within MERGE_DIST of each other, that
#      midpoint is a candidate node.
#   3. Candidate nodes get clustered: anything within CLUSTER_DIST is
#      fused into one node (so a true 3-way corner doesn't get counted
#      three times, once per line pair).
#   4. A node also needs PHYSICAL SUPPORT: enough cloud points within
#      SUPPORT_DIST of it. Without that test, nodes can land outside
#      the room (algebra is happy; physics isn't).

import os, numpy as np
from itertools import combinations

IN_PATH  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.npz"
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/nodes.npz"

MERGE_DIST     = 0.20   # two lines must approach within this distance to suggest a node
CLUSTER_DIST   = 0.30   # candidate nodes within this distance get fused
SUPPORT_DIST   = 0.25   # cloud points must lie within this distance of the node
MIN_SUPPORT    = 300


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 5 first)")
d = np.load(IN_PATH)
pts          = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]
lines_raw    = d["lines"]
print(f"loaded {len(lines_raw)} lines")

# lines_raw layout (from Cell 5): i j  point(3)  direction(3)  support  start(3)  end(3)  length
# rebuild structured arrays
line_pt  = lines_raw[:, 2:5]
line_dir = lines_raw[:, 5:8]
line_seg_s = lines_raw[:, 9:12]
line_seg_e = lines_raw[:, 12:15]
line_planes = lines_raw[:, :2].astype(int)
n_lines = len(line_pt)


def closest_approach(p1, d1, p2, d2):
    """Closest points on two infinite lines. Returns (midpoint, distance)."""
    w0 = p1 - p2
    a = d1 @ d1
    b = d1 @ d2
    c = d2 @ d2
    dd = d1 @ w0
    e  = d2 @ w0
    denom = a * c - b * b
    if abs(denom) < 1e-10:                  # parallel
        return None, np.inf
    s = (b * e - c * dd) / denom
    t = (a * e - b * dd) / denom
    closest1 = p1 + s * d1
    closest2 = p2 + t * d2
    mid = 0.5 * (closest1 + closest2)
    return mid, float(np.linalg.norm(closest1 - closest2))


# 1+2. collect candidate nodes from line pairs that approach each other
candidates = []   # (point, set of plane indices contributing)
for i, j in combinations(range(n_lines), 2):
    mid, dist = closest_approach(line_pt[i], line_dir[i],
                                  line_pt[j], line_dir[j])
    if mid is None or dist > MERGE_DIST:
        continue
    contributing_planes = set(line_planes[i].tolist() + line_planes[j].tolist())
    candidates.append((mid, contributing_planes))
print(f"  {len(candidates)} candidate nodes from line pairs (within {MERGE_DIST})")

# 3. cluster candidates (single-pass agglomerative by distance)
nodes = []        # list of (centroid, set of planes)
for cpt, cplanes in candidates:
    placed = False
    for k, (npt, nplanes) in enumerate(nodes):
        if np.linalg.norm(cpt - npt) < CLUSTER_DIST:
            # fuse: weighted centroid + union of planes
            new_pt = 0.5 * (npt + cpt)
            nodes[k] = (new_pt, nplanes | cplanes)
            placed = True
            break
    if not placed:
        nodes.append((cpt, set(cplanes)))
print(f"  {len(nodes)} nodes after clustering (CLUSTER_DIST={CLUSTER_DIST})")

# 4. physical support test
final = []
for (npt, nplanes) in nodes:
    dists = np.linalg.norm(pts - npt, axis=1)
    support = int((dists < SUPPORT_DIST).sum())
    if support < MIN_SUPPORT:
        continue
    final.append((npt, nplanes, support))
print(f"  {len(final)} nodes after physical-support filter (>={MIN_SUPPORT} pts)")

for k, (npt, nplanes, sup) in enumerate(final):
    print(f"    node {k}: pos=({npt[0]:+.2f},{npt[1]:+.2f},{npt[2]:+.2f})  "
          f"planes={sorted(nplanes)}  support={sup}")

# Save nodes for Cell 7
node_pts    = np.array([n[0] for n in final]) if final else np.zeros((0, 3))
node_planes = [sorted(n[1]) for n in final]
np.savez(OUT_NPZ,
         points=pts, plane_for_pt=plane_for_pt, planes=planes,
         lines=lines_raw, nodes=node_pts,
         node_plane_lists=np.array([str(p) for p in node_planes]))
print(f"\nsaved: {OUT_NPZ}")

loaded 12 lines
  42 candidate nodes from line pairs (within 0.2)
  26 nodes after clustering (CLUSTER_DIST=0.3)
  5 nodes after physical-support filter (>=300 pts)
    node 0: pos=(-0.52,-0.20,+1.77)  planes=[2, 4, 6]  support=8908
    node 1: pos=(-0.62,-0.73,+1.72)  planes=[0, 2, 4]  support=5510
    node 2: pos=(+0.11,+0.42,+2.21)  planes=[1, 2, 4, 5, 7]  support=5332
    node 3: pos=(-0.36,+0.44,+1.86)  planes=[1, 2, 4, 5, 7]  support=6968
    node 4: pos=(+0.65,+0.37,+2.49)  planes=[1, 4, 7]  support=1375

saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/nodes.npz


In [ ]:
# Cell 7 — Final skeleton: snap line endpoints to nodes, render the whole thing
# The skeleton graph is: nodes (corner points) connected by lines
# (intersections between planes). To produce a clean final figure, I
# snap each line's endpoints to the nearest node if a node sits close
# enough -- so adjoining edges meet at a corner instead of just passing
# near each other. This is the architectural cleanup step.
#
# Output: skeleton_final.ply
#   - cloud in light grey (background)
#   - planes in their assigned colours (faint)
#   - lines in red, snapped to nodes
#   - nodes as dense black point clusters (they're vertices)

import os, numpy as np

IN_PATH   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/nodes.npz"
OUT_PLY   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.ply"

SNAP_DIST          = 0.40    # if a line endpoint is within this of a node, snap to it
SAMPLES_PER_LINE   = 300
NODE_BALL_PTS      = 200     # how many points to draw per node (visibility)
NODE_BALL_RADIUS   = 0.04
PLANE_DIM_FACTOR   = 0.55    # planes are dimmed to background; lines/nodes pop


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 6 first)")
d = np.load(IN_PATH)
pts          = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]
lines_raw    = d["lines"]
nodes        = d["nodes"]
print(f"loaded {len(pts):,} points  | {len(planes)} planes  "
      f"| {len(lines_raw)} lines  | {len(nodes)} nodes")

# rebuild line endpoints
line_seg_s = lines_raw[:, 9:12].copy()
line_seg_e = lines_raw[:, 12:15].copy()


# --- snap each endpoint to nearest node if close enough ---
def snap_to_nearest(point, nodes, max_dist):
    if len(nodes) == 0:
        return point, False
    dists = np.linalg.norm(nodes - point, axis=1)
    k = int(np.argmin(dists))
    if dists[k] <= max_dist:
        return nodes[k], True
    return point, False

snapped_count = 0
for i in range(len(line_seg_s)):
    new_s, sn1 = snap_to_nearest(line_seg_s[i], nodes, SNAP_DIST)
    new_e, sn2 = snap_to_nearest(line_seg_e[i], nodes, SNAP_DIST)
    line_seg_s[i] = new_s
    line_seg_e[i] = new_e
    snapped_count += int(sn1) + int(sn2)
print(f"snapped {snapped_count} of {2*len(line_seg_s)} line endpoints to nodes")


# --- assemble PLY ---
# 1. planes as their colours, dimmed
PALETTE = np.array([
    [0.90, 0.30, 0.30], [0.30, 0.65, 0.90], [0.35, 0.80, 0.45],
    [0.95, 0.75, 0.20], [0.65, 0.40, 0.85], [0.95, 0.55, 0.20],
    [0.30, 0.80, 0.80], [0.85, 0.50, 0.70],
])
cloud_rgb = np.full((len(pts), 3), 0.85)
for k in range(len(planes)):
    mask = plane_for_pt == k
    cloud_rgb[mask] = PLANE_DIM_FACTOR * PALETTE[k % len(PALETTE)] + \
                       (1 - PLANE_DIM_FACTOR) * np.array([0.92, 0.92, 0.92])
cloud_255 = (cloud_rgb * 255).astype(np.uint8)

# 2. lines as red dense sample points
LINE_RGB = np.array([220, 30, 30], dtype=np.uint8)
line_xyz, line_rgb = [], []
for s, e in zip(line_seg_s, line_seg_e):
    ts = np.linspace(0, 1, SAMPLES_PER_LINE)
    line_xyz.append(s + ts[:, None] * (e - s))
    line_rgb.append(np.tile(LINE_RGB, (SAMPLES_PER_LINE, 1)))

# 3. nodes as black point clusters
NODE_RGB = np.array([0, 0, 0], dtype=np.uint8)
node_xyz, node_rgb = [], []
for n in nodes:
    # random points in a small ball around the node, for visibility
    rng = np.random.default_rng(0)
    dirs = rng.normal(size=(NODE_BALL_PTS, 3))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True) + 1e-9
    rs = rng.uniform(0, NODE_BALL_RADIUS, size=NODE_BALL_PTS)
    ball = n + rs[:, None] * dirs
    node_xyz.append(ball)
    node_rgb.append(np.tile(NODE_RGB, (NODE_BALL_PTS, 1)))

# stack everything
all_xyz = [pts]; all_rgb = [cloud_255]
if line_xyz: all_xyz.append(np.vstack(line_xyz)); all_rgb.append(np.vstack(line_rgb))
if node_xyz: all_xyz.append(np.vstack(node_xyz)); all_rgb.append(np.vstack(node_rgb))
all_xyz = np.vstack(all_xyz); all_rgb = np.vstack(all_rgb)

with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(all_xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(all_xyz, all_rgb):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"\nsaved: {OUT_PLY}")
print("\nopen it: planes faintly coloured, edges in red, corner nodes in black")

loaded 339,556 points  | 8 planes  | 12 lines  | 5 nodes
snapped 5 of 24 line endpoints to nodes

saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.ply

open it: planes faintly coloured, edges in red, corner nodes in black


In [ ]:
# Cell 8 — Manhattan-ness validation
# Before applying a Manhattan-world prior I check whether the room
# actually behaves like one. For every pair of planes I compute the
# angle between their normals, and report how many pairs are close to
# parallel (0 deg), close to perpendicular (90 deg), or off-axis.
# A high "Manhattan fraction" means the prior is empirically justified
# on this scene, not just assumed.

import os, numpy as np
from itertools import combinations

IN_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"

TOL_DEG = 15.0   # how close to 0 or 90 deg counts as aligned


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH} (run Cell 4 first)")
d = np.load(IN_PATH)
planes = d["planes"]
n = len(planes)
print(f"loaded {n} planes")

# normals + sizes
normals = planes[:, :3]
normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
sizes   = planes[:, 4]    # number of inlier points per plane

# pairwise angles
pairs = list(combinations(range(n), 2))
angles_deg = []
for i, j in pairs:
    cos_ang = abs(float(np.dot(normals[i], normals[j])))
    cos_ang = min(1.0, max(0.0, cos_ang))    # clip for safe arccos
    angles_deg.append(np.degrees(np.arccos(cos_ang)))
angles_deg = np.array(angles_deg)

# classify
is_parallel = angles_deg < TOL_DEG
is_perp     = np.abs(angles_deg - 90.0) < TOL_DEG
is_off      = ~is_parallel & ~is_perp

print(f"\npair-angle summary across {len(pairs)} plane pairs:")
print(f"  parallel       ({int(is_parallel.sum())}): angle within {TOL_DEG}° of 0°")
print(f"  perpendicular  ({int(is_perp.sum())}): angle within {TOL_DEG}° of 90°")
print(f"  off-axis       ({int(is_off.sum())}): neither")

frac = (is_parallel | is_perp).mean()
print(f"\nManhattan fraction = {frac:.0%}")

# weighted version: do BIG planes agree with the prior more than small ones?
# (this matters because small planes are often furniture artefacts)
pair_weights = np.array([sizes[i] * sizes[j] for i, j in pairs])
w_total = pair_weights.sum()
w_match = pair_weights[is_parallel | is_perp].sum()
frac_w  = w_match / w_total if w_total > 0 else 0.0
print(f"Manhattan fraction (size-weighted) = {frac_w:.0%}")

print()
if frac >= 0.70:
    print("=> room is approximately Manhattan-aligned; the prior is justified.")
elif frac >= 0.50:
    print("=> room is moderately Manhattan; regularise the aligned subset only.")
else:
    print("=> room is NOT Manhattan-like; do not apply the prior.")

loaded 8 planes

pair-angle summary across 28 plane pairs:
  parallel       (7): angle within 15.0° of 0°
  perpendicular  (15): angle within 15.0° of 90°
  off-axis       (6): neither

Manhattan fraction = 79%
Manhattan fraction (size-weighted) = 90%

=> room is approximately Manhattan-aligned; the prior is justified.


In [ ]:
# Diagnostic — inspect plane sizes, confidence and normals

import numpy as np, os

IN_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"
d = np.load(IN_PATH)
planes = d["planes"]

normals = planes[:, :3]
normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
sizes = planes[:, 4]
conf = planes[:, 5] if planes.shape[1] > 5 else np.full(len(planes), np.nan)

order = np.argsort(-sizes)

print("planes ranked by support:")
for idx in order:
    print(
        f"plane {idx}: "
        f"size={sizes[idx]:.0f}, "
        f"mean_conf={conf[idx]:.3f}, "
        f"normal={normals[idx]}"
    )

planes ranked by support:
plane 0: size=64527, mean_conf=0.614, normal=[0.05084158 0.99863483 0.01198357]
plane 1: size=49379, mean_conf=0.635, normal=[0.04003979 0.99914404 0.01039239]
plane 2: size=47618, mean_conf=0.432, normal=[-0.62059088  0.04941225  0.78257613]
plane 3: size=34389, mean_conf=0.442, normal=[ 0.77399903 -0.01481884  0.63301335]
plane 4: size=21526, mean_conf=0.544, normal=[-0.53205536  0.0259625   0.84631143]
plane 5: size=18590, mean_conf=0.642, normal=[ 0.15098719  0.97641128 -0.15434985]
plane 6: size=12775, mean_conf=0.551, normal=[-0.40057823  0.63231339  0.6631115 ]
plane 7: size=11487, mean_conf=0.654, normal=[ 0.09318177  0.9883599  -0.12025746]


In [ ]:
# Diagnostic — print all pair angles

from itertools import combinations
import numpy as np

TOL_DEG = 15.0

pairs_info = []
for i, j in combinations(range(len(normals)), 2):
    cos_ang = abs(float(normals[i] @ normals[j]))
    cos_ang = np.clip(cos_ang, 0, 1)
    ang = np.degrees(np.arccos(cos_ang))

    if ang < TOL_DEG:
        tag = "parallel"
    elif abs(ang - 90) < TOL_DEG:
        tag = "perpendicular"
    else:
        tag = "off-axis"

    weight = sizes[i] * sizes[j]
    pairs_info.append((i, j, ang, tag, weight))

pairs_info = sorted(pairs_info, key=lambda x: x[2])

for i, j, ang, tag, weight in pairs_info:
    print(f"planes {i}-{j}: angle={ang:5.1f}° | {tag:13s} | weight={weight:.2e}")

planes 0-1: angle=  0.6° | parallel      | weight=3.19e+09
planes 5-7: angle=  3.9° | parallel      | weight=2.14e+08
planes 2-4: angle=  6.4° | parallel      | weight=1.03e+09
planes 0-7: angle=  8.0° | parallel      | weight=7.41e+08
planes 1-7: angle=  8.1° | parallel      | weight=5.67e+08
planes 0-5: angle= 11.2° | parallel      | weight=1.20e+09
planes 1-5: angle= 11.5° | parallel      | weight=9.18e+08
planes 2-6: angle= 37.0° | off-axis      | weight=6.08e+08
planes 4-6: angle= 37.7° | off-axis      | weight=2.75e+08
planes 1-6: angle= 51.5° | off-axis      | weight=6.31e+08
planes 0-6: angle= 51.8° | off-axis      | weight=8.24e+08
planes 6-7: angle= 59.5° | off-axis      | weight=1.47e+08
planes 5-6: angle= 63.0° | off-axis      | weight=2.37e+08
planes 4-5: angle= 79.3° | perpendicular | weight=4.00e+08
planes 2-5: angle= 80.4° | perpendicular | weight=8.85e+08
planes 4-7: angle= 82.8° | perpendicular | weight=2.47e+08
planes 3-4: angle= 82.9° | perpendicular | weight=7.40e+

In [ ]:
# Cell 8b — Stricter Manhattan-ness test
# The earlier test used a 15 deg tolerance, which is generous on a
# small sample. Here I run three stricter checks:
#
#   (a) re-do the pairwise test with a 5 deg tolerance,
#   (b) cluster the plane normals into 3 axes (true Manhattan structure
#       has exactly 3 mutually orthogonal direction groups), and
#   (c) measure how far each plane's normal actually deviates from its
#       nearest cluster axis, weighted by plane size.
#
# Only if all three agree do I trust the Manhattan prior for this room.

import os, numpy as np
from itertools import combinations

IN_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"

TOL_STRICT = 5.0     # degrees: a much tighter pairwise tolerance
TOL_AXIS   = 5.0     # degrees: per-plane deviation from its cluster axis


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH}")
d = np.load(IN_PATH)
planes = d["planes"]
n = len(planes)

normals = planes[:, :3]
normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
sizes   = planes[:, 4]
print(f"loaded {n} planes (size range: {int(sizes.min())} .. {int(sizes.max())})")


# (a) -----------------------------------------------------------------
print(f"\n(a) STRICT PAIRWISE TEST (tolerance = {TOL_STRICT} deg)")
pairs = list(combinations(range(n), 2))
ang = []
for i, j in pairs:
    cos_ang = abs(float(np.dot(normals[i], normals[j])))
    cos_ang = min(1.0, max(0.0, cos_ang))
    ang.append(np.degrees(np.arccos(cos_ang)))
ang = np.array(ang)
par  = ang < TOL_STRICT
perp = np.abs(ang - 90.0) < TOL_STRICT
print(f"    parallel       : {int(par.sum())}/{len(pairs)}")
print(f"    perpendicular  : {int(perp.sum())}/{len(pairs)}")
print(f"    off-axis       : {int((~par & ~perp).sum())}/{len(pairs)}")
print(f"    fraction aligned at {TOL_STRICT} deg: {(par | perp).mean():.0%}")


# (b) -----------------------------------------------------------------
# Cluster the normals into 3 axes. I use a simple greedy approach:
# treat the largest plane's normal as axis 1; the largest remaining
# normal that is roughly perpendicular to axis 1 is axis 2; axis 3 is
# their cross product. This gives 3 candidate axes derived from the
# data itself (no hard-coded x/y/z).
print(f"\n(b) AXIS CLUSTERING (data-driven, not hard-coded x/y/z)")
order = np.argsort(-sizes)               # biggest first
axis1 = normals[order[0]]
# find next normal closest to perpendicular to axis1
axis2 = None
for k in order[1:]:
    cos_a = abs(float(np.dot(normals[k], axis1)))
    if cos_a < np.cos(np.radians(90 - 20)):     # roughly perpendicular
        axis2 = normals[k]
        # orthogonalise it w.r.t axis1, then normalise
        axis2 = axis2 - (axis2 @ axis1) * axis1
        axis2 = axis2 / np.linalg.norm(axis2)
        break
must(axis2 is not None, "no second axis nearly perpendicular to the first")
axis3 = np.cross(axis1, axis2)
axis3 = axis3 / np.linalg.norm(axis3)
axes = np.stack([axis1, axis2, axis3])
print(f"    axis 1: ({axis1[0]:+.2f}, {axis1[1]:+.2f}, {axis1[2]:+.2f})")
print(f"    axis 2: ({axis2[0]:+.2f}, {axis2[1]:+.2f}, {axis2[2]:+.2f})")
print(f"    axis 3: ({axis3[0]:+.2f}, {axis3[1]:+.2f}, {axis3[2]:+.2f})")

# now: for each plane, find the closest axis and the deviation in deg
dev_per_plane = []
nearest_axis_per_plane = []
for k in range(n):
    cosines = np.abs(axes @ normals[k])          # |dot| with each axis
    best = int(np.argmax(cosines))
    deg  = np.degrees(np.arccos(min(1.0, cosines[best])))
    dev_per_plane.append(deg)
    nearest_axis_per_plane.append(best)
dev_per_plane = np.array(dev_per_plane)
nearest_axis_per_plane = np.array(nearest_axis_per_plane)


# (c) -----------------------------------------------------------------
print(f"\n(c) PER-PLANE DEVIATION FROM ITS NEAREST AXIS")
print(f"    {'plane':>5}  {'size':>7}  {'axis':>5}  {'dev_deg':>8}")
for k in range(n):
    print(f"    {k:>5}  {int(sizes[k]):>7}  {nearest_axis_per_plane[k]:>5}  "
          f"{dev_per_plane[k]:>8.2f}")

# summary statistics
mean_dev = dev_per_plane.mean()
weighted_mean_dev = float((dev_per_plane * sizes).sum() / sizes.sum())
max_dev  = dev_per_plane.max()
under_tol = (dev_per_plane < TOL_AXIS).mean()

print(f"\n    mean deviation               : {mean_dev:.2f} deg")
print(f"    size-weighted mean deviation : {weighted_mean_dev:.2f} deg")
print(f"    max deviation                : {max_dev:.2f} deg")
print(f"    fraction within {TOL_AXIS} deg of an axis : {under_tol:.0%}")


# -- verdict ----------------------------------------------------------
print("\n=== VERDICT ===")
strict_pair_ok = (par | perp).mean() >= 0.60
axes_ok        = weighted_mean_dev <= 7.0
no_outlier     = max_dev <= 15.0

print(f"  strict pairwise alignment (>=60% at {TOL_STRICT} deg) : "
      f"{'OK' if strict_pair_ok else 'FAIL'}")
print(f"  size-weighted axis deviation (<=7 deg)              : "
      f"{'OK' if axes_ok else 'FAIL'}")
print(f"  no plane more than 15 deg off-axis                 : "
      f"{'OK' if no_outlier else 'FAIL'}")

if strict_pair_ok and axes_ok and no_outlier:
    print("\n  => All three strict tests pass. Apply Manhattan regularisation.")
elif axes_ok:
    print("\n  => The dominant structure is Manhattan but some smaller planes")
    print("     are off-axis. Safe to regularise BIG planes only.")
else:
    print("\n  => Not Manhattan-aligned strictly. Do NOT apply the prior naively.")

loaded 8 planes (size range: 11487 .. 64527)

(a) STRICT PAIRWISE TEST (tolerance = 5.0 deg)
    parallel       : 2/28
    perpendicular  : 9/28
    off-axis       : 17/28
    fraction aligned at 5.0 deg: 39%

(b) AXIS CLUSTERING (data-driven, not hard-coded x/y/z)
    axis 1: (+0.05, +1.00, +0.01)
    axis 2: (-0.62, +0.02, +0.78)
    axis 3: (+0.78, -0.05, +0.62)

(c) PER-PLANE DEVIATION FROM ITS NEAREST AXIS
    plane     size   axis   dev_deg
        0    64527      0      0.00
        1    49379      0      0.63
        2    47618      1      1.56
        3    34389      2      2.00
        4    21526      1      6.33
        5    18590      0     11.21
        6    12775      1     38.53
        7    11487      0      7.98

    mean deviation               : 8.53 deg
    size-weighted mean deviation : 4.24 deg
    max deviation                : 38.53 deg
    fraction within 5.0 deg of an axis : 50%

=== VERDICT ===
  strict pairwise alignment (>=60% at 5.0 deg) : FAIL
  size-weig

In [ ]:
# Cell 9 — Selective Manhattan regularisation
# I apply the prior ONLY to planes that are already close to one of the
# three data-driven axes (deviation < SNAP_TOL). Furniture-like planes
# that sit at odd angles are left untouched. The output is a set of
# regularised plane equations that I use as input to Cell 10 (re-running
# line + node extraction on cleaner geometry).
#
# Built-in checks (the rigour layer):
#   - report the rotation angle applied to each plane,
#   - confirm the residual deviation is 0 (planes are now exactly on axis),
#   - confirm the plane offsets (d) are unchanged (only normals snap;
#     positions stay where the data put them).

import os, numpy as np
from itertools import combinations

IN_PATH  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes.npz"
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes_manhattan.npz"

SNAP_TOL = 5.0      # only snap planes whose normal is within this many deg of an axis


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH}")
d = np.load(IN_PATH)
planes       = d["planes"].copy()
points       = d["points"]
plane_for_pt = d["plane_for_pt"]
n = len(planes)

normals = planes[:, :3] / np.linalg.norm(planes[:, :3], axis=1, keepdims=True)
sizes   = planes[:, 4]

# rebuild the three data-driven axes (same recipe as Cell 8b)
order = np.argsort(-sizes)
axis1 = normals[order[0]]
axis2 = None
for k in order[1:]:
    if abs(float(normals[k] @ axis1)) < np.cos(np.radians(70)):
        axis2 = normals[k] - (normals[k] @ axis1) * axis1
        axis2 = axis2 / np.linalg.norm(axis2)
        break
must(axis2 is not None, "could not find a second axis")
axis3 = np.cross(axis1, axis2); axis3 /= np.linalg.norm(axis3)
axes = np.stack([axis1, axis2, axis3])
print("axes (data-driven):")
for k, ax in enumerate(axes):
    print(f"  axis {k}: ({ax[0]:+.3f}, {ax[1]:+.3f}, {ax[2]:+.3f})")


# decide which planes to snap, and snap them
print(f"\nplane snapping (tolerance = {SNAP_TOL} deg):")
print(f"  {'plane':>5} {'size':>7} {'dev_before':>10} {'axis':>5} {'action':>12} {'dev_after':>10}")

snapped_planes = planes.copy()       # rows: a, b, c, d, size, mean_conf
rotation_log   = []                  # to inspect later

for k in range(n):
    n_k = normals[k]
    # find nearest axis (account for sign: |dot|)
    dots = axes @ n_k
    best = int(np.argmax(np.abs(dots)))
    sign = float(np.sign(dots[best])) or 1.0
    target = sign * axes[best]
    deviation = float(np.degrees(np.arccos(min(1.0, abs(float(n_k @ target))))))

    if deviation <= SNAP_TOL:
        # snap the normal to the axis; keep d so that the plane still
        # passes through (approximately) the centroid of its inliers.
        inlier_mask = (plane_for_pt == k)
        if inlier_mask.sum() > 0:
            centroid = points[inlier_mask].mean(axis=0)
            new_d = -float(target @ centroid)
        else:
            new_d = planes[k, 3]
        snapped_planes[k, :3] = target
        snapped_planes[k, 3]  = new_d
        rotation_log.append((k, deviation, 0.0, True))
        print(f"  {k:>5} {int(sizes[k]):>7} {deviation:>10.2f} {best:>5} {'SNAPPED':>12} {0.00:>10.2f}")
    else:
        rotation_log.append((k, deviation, deviation, False))
        print(f"  {k:>5} {int(sizes[k]):>7} {deviation:>10.2f} {best:>5} {'left as-is':>12} {deviation:>10.2f}")


# --- built-in validation checks --------------------------------------
print("\n--- validation ---")

# (i) Every snapped plane's residual deviation must be ~0
snapped_idx = [r[0] for r in rotation_log if r[3]]
print(f"  ({len(snapped_idx)} planes were snapped, "
      f"{n - len(snapped_idx)} left as-is)")

residuals = []
for k in snapped_idx:
    new_n = snapped_planes[k, :3]
    dots_after = np.abs(axes @ new_n)
    res = float(np.degrees(np.arccos(min(1.0, float(dots_after.max())))))
    residuals.append(res)
print(f"  residual deviation of snapped planes: max = {max(residuals) if residuals else 0:.4f} deg "
      f"(expected ~0)")

# (ii) Pairwise check: snapped planes should now be exactly 0 or 90 deg apart
if len(snapped_idx) >= 2:
    bad = 0
    for i, j in combinations(snapped_idx, 2):
        cos_ang = abs(float(snapped_planes[i, :3] @ snapped_planes[j, :3]))
        cos_ang = min(1.0, max(0.0, cos_ang))
        ang = float(np.degrees(np.arccos(cos_ang)))
        if not (ang < 0.5 or abs(ang - 90.0) < 0.5):
            bad += 1
    print(f"  pairwise post-snap: {bad} pair(s) NOT exactly 0/90 deg "
          f"out of {len(snapped_idx)*(len(snapped_idx)-1)//2}")

# (iii) Plane offsets ('d') change is reported (they re-anchor at centroid)
shifts = []
for k in snapped_idx:
    shifts.append(abs(float(snapped_planes[k, 3] - planes[k, 3])))
if shifts:
    print(f"  plane offset shifts (d): mean = {np.mean(shifts):.4f}, max = {max(shifts):.4f}")
    print(f"  (small shifts are healthy; they re-anchor the plane on its centroid)")


# save
np.savez(OUT_NPZ,
         points=points, plane_for_pt=plane_for_pt,
         planes=snapped_planes, axes=axes,
         snapped_idx=np.array(snapped_idx, dtype=int))
print(f"\nsaved: {OUT_NPZ}")
print(f"  contains: regularised planes (snapped + unchanged), 3 axes, snapped indices")

axes (data-driven):
  axis 0: (+0.051, +0.999, +0.012)
  axis 1: (-0.622, +0.022, +0.783)
  axis 2: (+0.781, -0.047, +0.622)

plane snapping (tolerance = 5.0 deg):
  plane    size dev_before  axis       action  dev_after
      0   64527       0.00     0      SNAPPED       0.00
      1   49379       0.63     0      SNAPPED       0.00
      2   47618       1.56     1      SNAPPED       0.00
      3   34389       2.00     2      SNAPPED       0.00
      4   21526       6.33     1   left as-is       6.33
      5   18590      11.21     0   left as-is      11.21
      6   12775      38.53     1   left as-is      38.53
      7   11487       7.98     0   left as-is       7.98

--- validation ---
  (4 planes were snapped, 4 left as-is)
  residual deviation of snapped planes: max = 0.0000 deg (expected ~0)
  pairwise post-snap: 0 pair(s) NOT exactly 0/90 deg out of 6
  plane offset shifts (d): mean = 0.0042, max = 0.0076
  (small shifts are healthy; they re-anchor the plane on its centroid)

sav

In [ ]:
# Cell 10 — Regenerate skeleton from Manhattan-regularised planes
# Re-runs the line and node extraction (same logic as Cells 5 and 6) on
# the regularised plane set, then compares the new skeleton to the
# previous (unregularised) one using metrics that reflect QUALITY, not
# just count:
#
#   - mean and median line length      (longer = less fragmented)
#   - number of short fragments (<0.30) (fewer = cleaner)
#   - mean support per line             (higher = each line is better backed)
#   - line length variance              (lower = more homogeneous, healthier)
#   - number of corner nodes            (informational, not a success metric)
#
# A cleaner skeleton may have FEWER lines than before -- that's fine
# and often expected. What matters is whether the surviving lines are
# longer, better supported, and less variable.

import os, numpy as np
from itertools import combinations

MAN_PATH       = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/planes_manhattan.npz"
PREV_SKEL_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton.npz"  # from Cell 5
PREV_NODE_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/nodes.npz"     # from Cell 6
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_manhattan.npz"
OUT_PLY  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_manhattan.ply"

# line-extraction settings (same scale as Cell 5)
PARALLEL_TOL      = 0.96
SUPPORT_DIST      = 0.10
MIN_SUPPORT_PTS   = 300
SEG_EXTENT_QUANT  = (0.005, 0.995)
MAX_LINES_KEPT    = 12
MIN_LINE_LENGTH   = 0.20

# node-extraction settings (same scale as Cell 6)
MERGE_DIST     = 0.20
CLUSTER_DIST   = 0.30
NODE_SUPPORT_D = 0.25
MIN_NODE_SUP   = 300

# snap line endpoints to nodes for the final figure
SNAP_DIST = 0.40
SAMPLES_PER_LINE = 300
NODE_BALL_PTS    = 200
NODE_BALL_RADIUS = 0.04


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")


# --- load ------------------------------------------------------------
must(os.path.isfile(MAN_PATH), f"missing: {MAN_PATH} (run Cell 9 first)")
d = np.load(MAN_PATH)
points       = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]
snapped_idx  = set(d["snapped_idx"].tolist())
n = len(planes)
print(f"loaded {n} planes ({len(snapped_idx)} regularised, {n-len(snapped_idx)} unchanged)")


# --- line extraction (same as Cell 5) --------------------------------
def intersect(p1, p2):
    n1, n2 = p1[:3], p2[:3]
    d1, d2 = p1[3], p2[3]
    direction = np.cross(n1, n2)
    norm = np.linalg.norm(direction)
    if norm < 1e-6: return None
    direction /= norm
    cos_ang = abs(float(n1 @ n2 / (np.linalg.norm(n1)*np.linalg.norm(n2))))
    if cos_ang > PARALLEL_TOL: return None
    A = np.array([n1, n2, direction]); b = np.array([-d1, -d2, 0.0])
    try: point = np.linalg.solve(A, b)
    except np.linalg.LinAlgError: return None
    return point, direction


candidates = []
for i, j in combinations(range(n), 2):
    r = intersect(planes[i], planes[j])
    if r is None: continue
    point, direction = r
    mask = (plane_for_pt == i) | (plane_for_pt == j)
    if mask.sum() == 0: continue
    v = points[mask] - point
    proj = v @ direction
    perp = v - np.outer(proj, direction)
    near = np.linalg.norm(perp, axis=1) < SUPPORT_DIST
    sup = int(near.sum())
    if sup < MIN_SUPPORT_PTS: continue
    t_lo, t_hi = np.quantile(proj[near], SEG_EXTENT_QUANT)
    s, e = point + t_lo * direction, point + t_hi * direction
    L = float(np.linalg.norm(e - s))
    if L < MIN_LINE_LENGTH: continue
    candidates.append((i, j, s, e, L, sup, point, direction))

candidates.sort(key=lambda x: x[4], reverse=True)
lines = candidates[:MAX_LINES_KEPT]
print(f"\nkept {len(lines)} lines after filtering")


# --- node extraction (same as Cell 6) --------------------------------
def closest_approach(p1, d1, p2, d2):
    w0 = p1 - p2
    a, b, c = d1 @ d1, d1 @ d2, d2 @ d2
    dd, e = d1 @ w0, d2 @ w0
    denom = a * c - b * b
    if abs(denom) < 1e-10: return None, np.inf
    s = (b * e - c * dd) / denom
    t = (a * e - b * dd) / denom
    c1, c2 = p1 + s * d1, p2 + t * d2
    return 0.5 * (c1 + c2), float(np.linalg.norm(c1 - c2))

cands = []
for ii, jj in combinations(range(len(lines)), 2):
    li, lj = lines[ii], lines[jj]
    p1, d1 = li[6], li[7]
    p2, d2 = lj[6], lj[7]
    mid, dist = closest_approach(p1, d1, p2, d2)
    if mid is None or dist > MERGE_DIST: continue
    cands.append(mid)

nodes_clustered = []
for cpt in cands:
    placed = False
    for k, npt in enumerate(nodes_clustered):
        if np.linalg.norm(cpt - npt) < CLUSTER_DIST:
            nodes_clustered[k] = 0.5 * (npt + cpt); placed = True; break
    if not placed:
        nodes_clustered.append(cpt)

nodes = []
for npt in nodes_clustered:
    sup = int((np.linalg.norm(points - npt, axis=1) < NODE_SUPPORT_D).sum())
    if sup >= MIN_NODE_SUP:
        nodes.append(npt)
nodes = np.array(nodes) if nodes else np.zeros((0, 3))
print(f"kept {len(nodes)} corner nodes")


# --- VALIDATION: compare with previous (unregularised) skeleton ------
print("\n=== before/after comparison ===")

def line_stats(lengths, supports):
    if len(lengths) == 0:
        return dict(n=0, mean_L=0, med_L=0, short=0, mean_sup=0, var_L=0)
    lengths = np.asarray(lengths); supports = np.asarray(supports)
    return dict(
        n        = len(lengths),
        mean_L   = float(lengths.mean()),
        med_L    = float(np.median(lengths)),
        short    = int((lengths < 0.30).sum()),
        mean_sup = float(supports.mean()),
        var_L    = float(lengths.var()),
    )

new_lengths  = [L for (_,_,_,_,L,_,_,_) in lines]
new_supports = [s for (_,_,_,_,_,s,_,_) in lines]
new_stats = line_stats(new_lengths, new_supports)

# previous skeleton (Cell 5 output)
prev_stats = None
if os.path.isfile(PREV_SKEL_PATH):
    pd = np.load(PREV_SKEL_PATH)
    if "lines" in pd.files and len(pd["lines"]) > 0:
        # layout (Cell 5): i j  point(3)  direction(3)  support  start(3)  end(3)  length
        prev = pd["lines"]
        prev_lengths  = prev[:, -1] if prev.shape[1] >= 16 else \
                         np.linalg.norm(prev[:, 12:15] - prev[:, 9:12], axis=1)
        prev_supports = prev[:, 8]
        prev_stats = line_stats(prev_lengths, prev_supports)

# previous nodes (Cell 6 output)
prev_n_nodes = 0
if os.path.isfile(PREV_NODE_PATH):
    nd = np.load(PREV_NODE_PATH)
    if "nodes" in nd.files:
        prev_n_nodes = len(nd["nodes"])

def fmt(stats, prev=None, key=None, more_is_better=True):
    if stats is None or stats["n"] == 0:
        return "  (no previous skeleton found)"
    v = stats[key]
    if prev is None:
        return f"{v:.3f}"
    p = prev[key]
    arrow = ""
    if more_is_better:
        arrow = " ↑" if v > p else (" ↓" if v < p else " =")
    else:
        arrow = " ↓ (good)" if v < p else (" ↑ (worse)" if v > p else " =")
    return f"{v:.3f}  (was {p:.3f}){arrow}"

print(f"{'metric':<30}{'manhattan':<28}{'previous':<22}")
print("-" * 80)
print(f"{'planes used':<30}{n:<28}{n:<22}")
print(f"{'lines kept':<30}{new_stats['n']:<28}"
      f"{prev_stats['n'] if prev_stats else '?':<22}")
print(f"{'mean line length':<30}"
      f"{new_stats['mean_L']:<28.3f}"
      f"{prev_stats['mean_L'] if prev_stats else 0:<22.3f}")
print(f"{'median line length':<30}"
      f"{new_stats['med_L']:<28.3f}"
      f"{prev_stats['med_L'] if prev_stats else 0:<22.3f}")
print(f"{'short fragments (<0.30)':<30}"
      f"{new_stats['short']:<28}"
      f"{prev_stats['short'] if prev_stats else '?':<22}")
print(f"{'mean support per line':<30}"
      f"{new_stats['mean_sup']:<28.0f}"
      f"{prev_stats['mean_sup'] if prev_stats else 0:<22.0f}")
print(f"{'line-length variance':<30}"
      f"{new_stats['var_L']:<28.3f}"
      f"{prev_stats['var_L'] if prev_stats else 0:<22.3f}")
print(f"{'corner nodes':<30}{len(nodes):<28}{prev_n_nodes:<22}")

# success interpretation
print("\nquality criteria (your definition):")
def verdict(label, new_val, prev_val, better="up"):
    if prev_stats is None: return f"  {label}: no baseline"
    if better == "up":
        ok = new_val >= prev_val
    else:
        ok = new_val <= prev_val
    return f"  {label}: {'OK' if ok else 'no change/worse'}  ({prev_val:.2f} -> {new_val:.2f})"
if prev_stats:
    print(verdict("mean line length",   new_stats['mean_L'],   prev_stats['mean_L'], "up"))
    print(verdict("median line length", new_stats['med_L'],    prev_stats['med_L'],  "up"))
    print(verdict("short fragments",    new_stats['short'],    prev_stats['short'],  "down"))
    print(verdict("mean support",       new_stats['mean_sup'], prev_stats['mean_sup'],"up"))
    print(verdict("length variance",    new_stats['var_L'],    prev_stats['var_L'],  "down"))


# --- save data + final visualisation ---------------------------------
lines_arr = np.array(
    [(i, j, *p, *dr, sup, *s, *e, L) for (i,j,s,e,L,sup,p,dr) in lines],
    dtype=float)
np.savez(OUT_NPZ, points=points, plane_for_pt=plane_for_pt,
         planes=planes, lines=lines_arr, nodes=nodes)

# render (planes faintly coloured, lines red, nodes black)
PALETTE = np.array([
    [0.90,0.30,0.30],[0.30,0.65,0.90],[0.35,0.80,0.45],[0.95,0.75,0.20],
    [0.65,0.40,0.85],[0.95,0.55,0.20],[0.30,0.80,0.80],[0.85,0.50,0.70]])
cloud_rgb = np.full((len(points), 3), 0.85)
for k in range(n):
    cloud_rgb[plane_for_pt == k] = 0.55 * PALETTE[k % len(PALETTE)] + 0.45 * 0.92
cloud_255 = (cloud_rgb * 255).astype(np.uint8)

def snap_endpoint(pt, nodes, max_d):
    if len(nodes) == 0: return pt
    dists = np.linalg.norm(nodes - pt, axis=1)
    k = int(np.argmin(dists))
    return nodes[k] if dists[k] <= max_d else pt

line_xyz, line_rgb = [], []
LR = np.array([220, 30, 30], dtype=np.uint8)
for (_,_,s,e,_,_,_,_) in lines:
    s = snap_endpoint(s, nodes, SNAP_DIST)
    e = snap_endpoint(e, nodes, SNAP_DIST)
    ts = np.linspace(0, 1, SAMPLES_PER_LINE)
    line_xyz.append(s + ts[:, None] * (e - s))
    line_rgb.append(np.tile(LR, (SAMPLES_PER_LINE, 1)))

node_xyz, node_rgb = [], []
NR = np.array([0, 0, 0], dtype=np.uint8)
rng = np.random.default_rng(0)
for nd in nodes:
    dirs = rng.normal(size=(NODE_BALL_PTS, 3))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True) + 1e-9
    rs = rng.uniform(0, NODE_BALL_RADIUS, size=NODE_BALL_PTS)
    ball = nd + rs[:, None] * dirs
    node_xyz.append(ball); node_rgb.append(np.tile(NR, (NODE_BALL_PTS, 1)))

all_xyz = [points]; all_rgb = [cloud_255]
if line_xyz: all_xyz.append(np.vstack(line_xyz)); all_rgb.append(np.vstack(line_rgb))
if node_xyz: all_xyz.append(np.vstack(node_xyz)); all_rgb.append(np.vstack(node_rgb))
all_xyz = np.vstack(all_xyz); all_rgb = np.vstack(all_rgb)

with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(all_xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(all_xyz, all_rgb):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"\nsaved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}  (planes faint, lines red, nodes black)")

loaded 8 planes (4 regularised, 4 unchanged)

kept 12 lines after filtering
kept 5 corner nodes

=== before/after comparison ===
metric                        manhattan                   previous              
--------------------------------------------------------------------------------
planes used                   8                           8                     
lines kept                    12                          12                    
mean line length              2.097                       2.086                 
median line length            2.147                       2.090                 
short fragments (<0.30)       0                           0                     
mean support per line         6233                        6268                  
line-length variance          0.116                       0.115                 
corner nodes                  5                           5                     

quality criteria (your definition):
  mean line length: OK  

In [ ]:
# Cell 11 — RANSAC distance threshold sweep
# A controlled experiment: I sweep the RANSAC plane-fit threshold,
# rerun the full plane -> line pipeline at each value, and report
# the same quality metrics for each. I do NOT change other parameters
# during the sweep -- the experiment isolates the effect of this one
# choice. The "best" threshold is the one whose metrics beat the
# baseline (0.03) on the criteria we defined.

import os, numpy as np
from itertools import combinations
import open3d as o3d

CLEAN_PATH = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.npz"

THRESHOLDS = [0.02, 0.03, 0.04, 0.05, 0.06]    # the sweep

# everything below is held constant across the sweep
MAX_PLANES        = 8
MIN_PLANE_POINTS  = 2000
PARALLEL_TOL      = 0.96
SUPPORT_DIST      = 0.10
MIN_SUPPORT_PTS   = 300
SEG_EXTENT_QUANT  = (0.005, 0.995)
MAX_LINES_KEPT    = 12
MIN_LINE_LENGTH   = 0.20


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(CLEAN_PATH), f"missing: {CLEAN_PATH}")
d = np.load(CLEAN_PATH)
pts  = d["points"]
col  = d["colors"]
geom = d["geom_conf"]
print(f"loaded {len(pts):,} points")


def extract_planes(distance_threshold):
    """Run iterative RANSAC plane extraction and return the plane equations."""
    sub = o3d.geometry.PointCloud()
    sub.points = o3d.utility.Vector3dVector(pts)
    remaining = np.arange(len(pts))
    planes = []
    plane_for_pt = np.full(len(pts), -1, dtype=int)
    weights_all = geom / geom.sum() if geom.sum() > 0 else np.ones(len(pts))

    for k in range(MAX_PLANES):
        if len(remaining) < MIN_PLANE_POINTS:
            break
        cur = o3d.geometry.PointCloud()
        cur.points = o3d.utility.Vector3dVector(pts[remaining])
        # confidence-weighted scoring with 3 starts
        best_inliers, best_score, best_model = None, -1, None
        for _ in range(3):
            model, inliers = cur.segment_plane(
                distance_threshold=distance_threshold,
                ransac_n=3, num_iterations=2000)
            score = weights_all[remaining[inliers]].sum()
            if score > best_score:
                best_score = score
                best_inliers = inliers
                best_model = model
        if best_inliers is None or len(best_inliers) < MIN_PLANE_POINTS:
            break
        a, b, c, dconst = best_model
        orig = remaining[best_inliers]
        plane_for_pt[orig] = k
        planes.append((a, b, c, dconst, len(orig), float(geom[orig].mean())))
        mask = np.ones(len(remaining), dtype=bool); mask[best_inliers] = False
        remaining = remaining[mask]

    return np.array(planes, dtype=float), plane_for_pt


def intersect(p1, p2):
    n1, n2 = p1[:3], p2[:3]
    d1, d2 = p1[3], p2[3]
    direction = np.cross(n1, n2)
    norm = np.linalg.norm(direction)
    if norm < 1e-6: return None
    direction /= norm
    cos_ang = abs(float(n1 @ n2 / (np.linalg.norm(n1)*np.linalg.norm(n2))))
    if cos_ang > PARALLEL_TOL: return None
    A = np.array([n1, n2, direction]); b = np.array([-d1, -d2, 0.0])
    try: point = np.linalg.solve(A, b)
    except np.linalg.LinAlgError: return None
    return point, direction


def extract_lines(planes, plane_for_pt):
    n = len(planes)
    candidates = []
    for i, j in combinations(range(n), 2):
        r = intersect(planes[i], planes[j])
        if r is None: continue
        point, direction = r
        mask = (plane_for_pt == i) | (plane_for_pt == j)
        if mask.sum() == 0: continue
        v = pts[mask] - point
        proj = v @ direction
        perp = v - np.outer(proj, direction)
        near = np.linalg.norm(perp, axis=1) < SUPPORT_DIST
        sup = int(near.sum())
        if sup < MIN_SUPPORT_PTS: continue
        t_lo, t_hi = np.quantile(proj[near], SEG_EXTENT_QUANT)
        s = point + t_lo * direction; e = point + t_hi * direction
        L = float(np.linalg.norm(e - s))
        if L < MIN_LINE_LENGTH: continue
        candidates.append((L, sup))
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[:MAX_LINES_KEPT]


# -------- run the sweep ----------------------------------------------
print(f"\n{'threshold':>10} {'planes':>7} {'lines':>6} "
      f"{'mean_L':>8} {'med_L':>8} {'short':>6} {'mean_sup':>9} {'var_L':>8}")
print("-" * 70)

rows = []
for t in THRESHOLDS:
    planes_t, p4p = extract_planes(t)
    lines_t = extract_lines(planes_t, p4p)
    if len(lines_t) == 0:
        rows.append((t, len(planes_t), 0, 0, 0, 0, 0, 0))
        print(f"{t:>10.3f} {len(planes_t):>7} {0:>6} {0:>8} {0:>8} {0:>6} {0:>9} {0:>8}")
        continue
    Ls   = np.array([L for (L, _) in lines_t])
    Sups = np.array([s for (_, s) in lines_t])
    row = (t, len(planes_t), len(lines_t), Ls.mean(), np.median(Ls),
           int((Ls < 0.30).sum()), Sups.mean(), Ls.var())
    rows.append(row)
    print(f"{t:>10.3f} {row[1]:>7} {row[2]:>6} {row[3]:>8.3f} "
          f"{row[4]:>8.3f} {row[5]:>6} {row[6]:>9.0f} {row[7]:>8.3f}")


# -------- decide using pre-defined criteria --------------------------
print("\n--- baseline: threshold = 0.03 (the value we used so far)")
baseline = next(r for r in rows if abs(r[0] - 0.03) < 1e-6)
_, _, _, b_mean, b_med, b_short, b_sup, b_var = baseline

print("--- criteria (set BEFORE the sweep):")
print("    mean_L up, med_L up, short down, mean_sup up, var_L down")
print("--- which thresholds beat the baseline on at least 3 of 5 criteria?")
winners = []
for r in rows:
    if abs(r[0] - 0.03) < 1e-6:                # skip baseline
        continue
    _, _, _, mean, med, short, sup, var = r
    score = sum([
        mean >  b_mean,
        med  >  b_med,
        short <  b_short,
        sup  >  b_sup,
        var  <  b_var,
    ])
    print(f"    t={r[0]:.3f}: beats baseline on {score}/5")
    if score >= 3:
        winners.append((r[0], score))

print()
if winners:
    best_t = max(winners, key=lambda x: x[1])[0]
    print(f"=> winner: threshold = {best_t} (beats baseline on majority of criteria)")
else:
    print("=> no threshold beats baseline. Keep 0.03 and stop tuning.")

loaded 339,556 points

 threshold  planes  lines   mean_L    med_L  short  mean_sup    var_L
----------------------------------------------------------------------
     0.020       8     12    1.897    1.918      0      5026    0.203
     0.030       8     12    1.890    1.910      0      4774    0.174
     0.040       8     12    2.182    2.307      0      7083    0.142
     0.050       8     12    2.174    2.146      0      7748    0.073
     0.060       8     12    2.197    2.288      0      9685    0.123

--- baseline: threshold = 0.03 (the value we used so far)
--- criteria (set BEFORE the sweep):
    mean_L up, med_L up, short down, mean_sup up, var_L down
--- which thresholds beat the baseline on at least 3 of 5 criteria?
    t=0.020: beats baseline on 3/5
    t=0.040: beats baseline on 4/5
    t=0.050: beats baseline on 4/5
    t=0.060: beats baseline on 4/5

=> winner: threshold = 0.04 (beats baseline on majority of criteria)


In [ ]:
# Cell 12b — Corner nodes from TRIPLE plane intersections
# The previous node detector looked for places where two LINES come
# close. After Manhattan regularisation many lines are exactly
# parallel, so they never approach -- and the detector returned 0.
#
# The fix is to drop lines from the formulation entirely and solve
# directly for the point where THREE planes meet:
#
#     [n1] [x]     [-d1]
#     [n2] [y]  =  [-d2]
#     [n3] [z]     [-d3]
#
# That 3x3 system has a unique solution iff the three normals are
# linearly independent (which we test by determinant). The result is
# a clean architectural node candidate. Then we filter by:
#   (a) the candidate sits inside the scene bounding box (expanded
#       slightly so corners on the edge are kept),
#   (b) it has physical support: cloud points nearby.
# Finally we cluster near-duplicates and snap line endpoints to nodes.

import os, numpy as np
from itertools import combinations

IN_PATH  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.npz"
OUT_NPZ  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.npz"
OUT_PLY  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.ply"

# corner-detection settings
BBOX_EXPAND       = 0.20    # allow candidates this far outside the scene bbox
INDEPENDENCE_TOL  = 0.05    # |det(N)| must exceed this for the triple to be well-conditioned
SUPPORT_RADIUS    = 0.30    # a corner needs this many cloud points within this distance
MIN_SUPPORT       = 200
CLUSTER_DIST      = 0.30
SNAP_DIST         = 0.40

# rendering settings (same as before)
SAMPLES_PER_LINE  = 300
NODE_BALL_PTS     = 200
NODE_BALL_RADIUS  = 0.05


def must(c, m):
    if not c: raise SystemExit(f"[STOP] {m}")

must(os.path.isfile(IN_PATH), f"missing: {IN_PATH}")
d = np.load(IN_PATH)
pts          = d["points"]
plane_for_pt = d["plane_for_pt"]
planes       = d["planes"]
lines_arr    = d["lines"]
n = len(planes)
print(f"loaded {n} planes  | {len(lines_arr)} lines  | {len(pts):,} points")


# ---- compute scene bounding box (expanded) --------------------------
bbox_min = pts.min(axis=0) - BBOX_EXPAND
bbox_max = pts.max(axis=0) + BBOX_EXPAND
print(f"scene bbox: {pts.min(0).round(2)} -> {pts.max(0).round(2)}")
print(f"with margin {BBOX_EXPAND}: candidates must lie within this expanded box")


# ---- triple-plane intersections -------------------------------------
def intersect_three(p1, p2, p3):
    """Solve for the point where three planes meet. Returns None if ill-conditioned."""
    N = np.stack([p1[:3], p2[:3], p3[:3]])           # 3x3 normal matrix
    d = -np.array([p1[3], p2[3], p3[3]])
    det = float(np.linalg.det(N))
    if abs(det) < INDEPENDENCE_TOL:
        return None                                    # normals not independent enough
    try:
        return np.linalg.solve(N, d)
    except np.linalg.LinAlgError:
        return None

candidates = []      # (point, frozenset of plane indices)
triples_tested = 0
for i, j, k in combinations(range(n), 3):
    triples_tested += 1
    pt = intersect_three(planes[i], planes[j], planes[k])
    if pt is None:
        continue
    if np.any(pt < bbox_min) or np.any(pt > bbox_max):
        continue                                       # outside the room
    candidates.append((pt, frozenset((i, j, k))))
print(f"\ntriples tested: {triples_tested}")
print(f"candidates inside expanded bbox: {len(candidates)}")


# ---- physical support filter ----------------------------------------
supported = []
for pt, planes_set in candidates:
    sup = int((np.linalg.norm(pts - pt, axis=1) < SUPPORT_RADIUS).sum())
    if sup >= MIN_SUPPORT:
        supported.append((pt, planes_set, sup))
print(f"candidates with >= {MIN_SUPPORT} points within {SUPPORT_RADIUS}: {len(supported)}")


# ---- cluster nearby candidates --------------------------------------
nodes = []          # (point, set of plane indices, total support)
for pt, ps, sup in supported:
    placed = False
    for k_idx, (npt, np_set, nsup) in enumerate(nodes):
        if np.linalg.norm(pt - npt) < CLUSTER_DIST:
            # fuse: support-weighted centroid + union of plane indices
            w = nsup + sup
            new_pt = (npt * nsup + pt * sup) / w
            nodes[k_idx] = (new_pt, np_set | ps, w)
            placed = True
            break
    if not placed:
        nodes.append((pt, set(ps), sup))
print(f"after clustering (radius {CLUSTER_DIST}): {len(nodes)} nodes")

for k, (npt, ps, sup) in enumerate(nodes):
    print(f"  node {k}: pos=({npt[0]:+.2f},{npt[1]:+.2f},{npt[2]:+.2f})  "
          f"planes={sorted(ps)}  support={sup}")

node_pts = np.array([n_[0] for n_ in nodes]) if nodes else np.zeros((0, 3))


# ---- snap line endpoints to nearest node ----------------------------
# line layout: i j  point(3)  direction(3)  support  start(3)  end(3)  length
seg_s = lines_arr[:, 9:12].copy()
seg_e = lines_arr[:, 12:15].copy()

def snap(pt, nodes_pts, max_d):
    if len(nodes_pts) == 0:
        return pt, False
    di = np.linalg.norm(nodes_pts - pt, axis=1)
    k = int(np.argmin(di))
    return (nodes_pts[k], True) if di[k] <= max_d else (pt, False)

snapped = 0
for i in range(len(seg_s)):
    new_s, ok1 = snap(seg_s[i], node_pts, SNAP_DIST)
    new_e, ok2 = snap(seg_e[i], node_pts, SNAP_DIST)
    seg_s[i] = new_s; seg_e[i] = new_e
    snapped += int(ok1) + int(ok2)
print(f"\nline-endpoint snapping: {snapped}/{2*len(seg_s)} endpoints snapped to nodes")


# ---- save -----------------------------------------------------------
lines_updated = lines_arr.copy()
lines_updated[:, 9:12]  = seg_s
lines_updated[:, 12:15] = seg_e
# recompute lengths
lines_updated[:, -1] = np.linalg.norm(seg_e - seg_s, axis=1)

np.savez(OUT_NPZ, points=pts, plane_for_pt=plane_for_pt,
         planes=planes, lines=lines_updated, nodes=node_pts,
         ransac_threshold=d["ransac_threshold"])


# ---- render (same scheme as before) ---------------------------------
PALETTE = np.array([
    [0.90,0.30,0.30],[0.30,0.65,0.90],[0.35,0.80,0.45],[0.95,0.75,0.20],
    [0.65,0.40,0.85],[0.95,0.55,0.20],[0.30,0.80,0.80],[0.85,0.50,0.70]])
cloud_rgb = np.full((len(pts), 3), 0.85)
for kk in range(n):
    cloud_rgb[plane_for_pt == kk] = 0.55 * PALETTE[kk % len(PALETTE)] + 0.45 * 0.92
cloud_255 = (cloud_rgb * 255).astype(np.uint8)

extra_xyz, extra_rgb = [], []
LR = np.array([220, 30, 30], dtype=np.uint8)
for s, e in zip(seg_s, seg_e):
    ts = np.linspace(0, 1, SAMPLES_PER_LINE)
    extra_xyz.append(s + ts[:, None] * (e - s))
    extra_rgb.append(np.tile(LR, (SAMPLES_PER_LINE, 1)))

NR = np.array([0, 0, 0], dtype=np.uint8)
rng = np.random.default_rng(0)
for nd in node_pts:
    dirs = rng.normal(size=(NODE_BALL_PTS, 3))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True) + 1e-9
    rs = rng.uniform(0, NODE_BALL_RADIUS, size=NODE_BALL_PTS)
    extra_xyz.append(nd + rs[:, None] * dirs)
    extra_rgb.append(np.tile(NR, (NODE_BALL_PTS, 1)))

all_xyz = [pts]; all_rgb = [cloud_255]
if extra_xyz:
    all_xyz.append(np.vstack(extra_xyz)); all_rgb.append(np.vstack(extra_rgb))
all_xyz = np.vstack(all_xyz); all_rgb = np.vstack(all_rgb)

with open(OUT_PLY, "w") as f:
    f.write(f"ply\nformat ascii 1.0\nelement vertex {len(all_xyz)}\n"
            "property float x\nproperty float y\nproperty float z\n"
            "property uchar red\nproperty uchar green\nproperty uchar blue\n"
            "end_header\n")
    for (x, y, z), (r, g, b) in zip(all_xyz, all_rgb):
        f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")

print(f"\nsaved: {OUT_NPZ}")
print(f"saved: {OUT_PLY}")

loaded 8 planes  | 12 lines  | 339,556 points
scene bbox: [-1.51 -0.87  0.01] -> [1.45 0.53 2.63]
with margin 0.2: candidates must lie within this expanded box

triples tested: 56
candidates inside expanded bbox: 16
candidates with >= 200 points within 0.3: 12
after clustering (radius 0.3): 5 nodes
  node 0: pos=(+0.47,-0.67,+2.62)  planes=[0, 1, 3, 4, 6]  support=24078
  node 1: pos=(+0.56,+0.32,+2.58)  planes=[1, 2, 3, 7]  support=1151
  node 2: pos=(+0.52,-0.43,+2.57)  planes=[1, 3, 6]  support=10946
  node 3: pos=(+0.98,+0.33,+2.04)  planes=[2, 3, 5, 6, 7]  support=33662
  node 4: pos=(+1.32,+0.24,+1.59)  planes=[3, 5, 7]  support=5010

line-endpoint snapping: 7/24 endpoints snapped to nodes

saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.npz
saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/skeleton_final.ply


In [1]:
import os

# 1. Mount Drive (au cas où Colab a redémarré)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2. Vérifier les fichiers principaux
BASE = "/content/drive/MyDrive/Humanoid_3D_Reconstruction"
OUT  = f"{BASE}/mast3r_output"

print("=== Tes fichiers de sortie ===\n")
files_to_check = [
    f"{BASE}/videos/challenge.mp4",
    f"{OUT}/scene.npz",
    f"{OUT}/scene.ply",
    f"{OUT}/clean.npz",
    f"{OUT}/clean.ply",
    f"{OUT}/planes_final.npz",
    f"{OUT}/skeleton_final.npz",
    f"{OUT}/skeleton_final.ply",
]
for f in files_to_check:
    if os.path.isfile(f):
        size_mb = os.path.getsize(f) / 1e6
        print(f"  ✅ {os.path.basename(f):30}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ MANQUANT: {os.path.basename(f)}")

print("\n=== Frames sélectionnés ===")
frames = "/content/selected_frames"
if os.path.isdir(frames):
    n = len([f for f in os.listdir(frames) if f.endswith('.jpg')])
    print(f"  ✅ {n} frames dans /content/selected_frames")
else:
    print("  ⚠️  /content/selected_frames vide (Colab redémarré)")
    print("     → on les regénérera si besoin")

Mounted at /content/drive
=== Tes fichiers de sortie ===

  ✅ challenge.mp4                   (49.7 MB)
  ✅ scene.npz                       (165.2 MB)
  ✅ scene.ply                       (410.4 MB)
  ✅ clean.npz                       (21.7 MB)
  ✅ clean.ply                       (23.6 MB)
  ✅ planes_final.npz                (10.9 MB)
  ✅ skeleton_final.npz              (10.9 MB)
  ✅ skeleton_final.ply              (24.2 MB)

=== Frames sélectionnés ===
  ⚠️  /content/selected_frames vide (Colab redémarré)
     → on les regénérera si besoin


In [4]:
# Cell — Generate the four README figures in one pass.
import os, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

OUT_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output"
FIG_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/figures"
CLEAN_NPZ = f"{OUT_DIR}/clean.npz"
SKEL_PLY  = f"{OUT_DIR}/skeleton_final.ply"

os.makedirs(FIG_DIR, exist_ok=True)

# --- Figure 1: skeleton_final.png ---
with open(SKEL_PLY) as f:
    skip = 0
    for line in f:
        skip += 1
        if line.strip() == "end_header": break
data = np.loadtxt(SKEL_PLY, skiprows=skip)
xyz = data[:, :3]; rgb = data[:, 3:6].astype(int)
is_red   = (rgb[:,0] > 180) & (rgb[:,2] < 80)
is_black = (rgb[:,0] < 30) & (rgb[:,1] < 30) & (rgb[:,2] < 30)
line_pts  = xyz[is_red]; node_pts = xyz[is_black]
cloud_pts = xyz[~is_red & ~is_black]; cloud_rgb = rgb[~is_red & ~is_black]/255.0
if len(cloud_pts) > 70000:
    sel = np.random.choice(len(cloud_pts), 70000, replace=False)
    cloud_pts = cloud_pts[sel]; cloud_rgb = cloud_rgb[sel]

fig = plt.figure(figsize=(16, 5))
for k, (e, a, t) in enumerate([(20,-60,"perspective 1"),(20,30,"perspective 2"),(5,0,"front"),(85,-90,"top-down")]):
    ax = fig.add_subplot(1, 4, k+1, projection="3d")
    ax.scatter(cloud_pts[:,0], cloud_pts[:,1], cloud_pts[:,2], c=cloud_rgb, s=0.6, depthshade=False)
    if len(line_pts): ax.scatter(line_pts[:,0], line_pts[:,1], line_pts[:,2], c="#cc1a1a", s=2.5, depthshade=False)
    if len(node_pts): ax.scatter(node_pts[:,0], node_pts[:,1], node_pts[:,2], c="k", s=14, depthshade=False)
    ax.view_init(elev=e, azim=a); ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
fig.suptitle("Final skeleton: planes (faint), edges (red), corner nodes (black)", fontsize=11)
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig(f"{FIG_DIR}/skeleton_final.png", dpi=140, bbox_inches="tight")
plt.close()
print("[ok] skeleton_final.png")

# --- Figure 2: confidence_heatmap.png ---
d = np.load(CLEAN_NPZ)
pts = d["points"]; gc_score = d["geom_conf"]
if len(pts) > 70000:
    sel = np.random.choice(len(pts), 70000, replace=False)
    pts_s = pts[sel]; gc_s = gc_score[sel]
else:
    pts_s = pts; gc_s = gc_score
cmap = plt.get_cmap("RdYlGn")
colors_rgb = cmap(np.clip(gc_s, 0, 1))[:, :3]

fig = plt.figure(figsize=(14, 5))
for k, (e, a, t) in enumerate([(20,-60,"perspective"),(85,-90,"top-down")]):
    ax = fig.add_subplot(1, 2, k+1, projection="3d")
    ax.scatter(pts_s[:,0], pts_s[:,1], pts_s[:,2], c=colors_rgb, s=0.7, depthshade=False)
    ax.view_init(elev=e, azim=a); ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1)); sm.set_array([])
fig.colorbar(sm, ax=fig.axes, fraction=0.03, pad=0.04).set_label("geometric confidence", fontsize=9)
fig.suptitle("Per-point geometric confidence (SVD planarity)", fontsize=11)
plt.savefig(f"{FIG_DIR}/confidence_heatmap.png", dpi=140, bbox_inches="tight")
plt.close()
print("[ok] confidence_heatmap.png")

# --- Figure 3: ransac_sweep.png ---
thresholds = np.array([0.020, 0.030, 0.040, 0.050, 0.060])
mean_L = np.array([1.897, 1.890, 2.182, 2.174, 2.197])
median_L = np.array([1.918, 1.910, 2.307, 2.146, 2.288])
mean_sup = np.array([5026, 4774, 7083, 7748, 9685])
var_L = np.array([0.203, 0.174, 0.142, 0.073, 0.123])
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
metrics = [("mean line length", mean_L, "up"),("median line length", median_L, "up"),
           ("mean support / line", mean_sup, "up"),("line-length variance", var_L, "down")]
for ax, (label, vals, dir_) in zip(axes, metrics):
    bars = ax.bar(thresholds, vals, width=0.008, color=["#bbbbbb"]*5)
    bars[2].set_color("#1D9E75")
    bars[1].set_edgecolor("k"); bars[1].set_linewidth(1.8)
    ax.set_xticks(thresholds); ax.set_xticklabels([f"{t:.02f}" for t in thresholds], fontsize=8)
    ax.set_xlabel("RANSAC threshold", fontsize=9)
    ax.set_title(f"{label}\n({dir_} is better)", fontsize=10)
    for s in ("top","right"): ax.spines[s].set_visible(False)
fig.suptitle("RANSAC sweep — 0.04 wins (green); baseline 0.03 outlined", fontsize=11)
plt.tight_layout(rect=[0,0,1,0.93])
plt.savefig(f"{FIG_DIR}/ransac_sweep.png", dpi=140, bbox_inches="tight")
plt.close()
print("[ok] ransac_sweep.png")

# --- Figure 4: pipeline.png ---
fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis("off")
blocks = [(0.3,"Phone video\n964 frames"),(2.3,"Frame selection\n(arc-length, 40)"),
          (4.6,"MASt3R\nreconstruction"),(7.1,"Clean + SVD\nconfidence"),
          (9.7,"Planes\n(RANSAC, weighted)"),(12.2,"Skeleton\nlines + nodes")]
colors_box = ["#dddddd","#cce0ff","#ffd9b3","#c8e6c9","#b5d8e8","#f9d2e0"]
for (x, label), c in zip(blocks, colors_box):
    ax.add_patch(FancyBboxPatch((x-0.55, 1.6), 1.6, 0.95, boxstyle="round,pad=0.05", facecolor=c, edgecolor="k", linewidth=1))
    ax.text(x+0.25, 2.07, label, ha="center", va="center", fontsize=9)
for i in range(len(blocks)-1):
    ax.add_patch(FancyArrowPatch((blocks[i][0]+1.1, 2.07), (blocks[i+1][0]-0.6, 2.07), arrowstyle="->", mutation_scale=15, color="k", linewidth=1.2))
ax.text(6.5, 3.7, "Pipeline overview — each stage has a validation check", ha="center", fontsize=11, fontweight="bold")
plt.savefig(f"{FIG_DIR}/pipeline.png", dpi=140, bbox_inches="tight")
plt.close()
print("[ok] pipeline.png")

print(f"\n✅ Toutes les figures dans : {FIG_DIR}")

[ok] skeleton_final.png
[ok] confidence_heatmap.png
[ok] ransac_sweep.png
[ok] pipeline.png

✅ Toutes les figures dans : /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures


In [5]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CLEAN_PLY = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output/clean.ply"
FIG_DIR   = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/figures"

with open(CLEAN_PLY) as f:
    skip = 0
    for line in f:
        skip += 1
        if line.strip() == "end_header": break
data = np.loadtxt(CLEAN_PLY, skiprows=skip)
xyz = data[:, :3]; rgb = data[:, 3:6] / 255.0
print(f"loaded {len(xyz):,} points")

if len(xyz) > 80000:
    sel = np.random.choice(len(xyz), 80000, replace=False)
    xyz = xyz[sel]; rgb = rgb[sel]

fig = plt.figure(figsize=(16, 5))
for k, (e, a, t) in enumerate([(20,-60,"perspective 1"),
                                (20, 30,"perspective 2"),
                                ( 5,  0,"front elevation"),
                                (85,-90,"top-down")]):
    ax = fig.add_subplot(1, 4, k+1, projection="3d")
    ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=rgb, s=0.7, depthshade=False)
    ax.view_init(elev=e, azim=a); ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

fig.suptitle("Dense 3D reconstruction (MASt3R, 339K points after cleaning)", fontsize=11)
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig(f"{FIG_DIR}/reconstruction.png", dpi=140, bbox_inches="tight")
plt.close()
print(f"✅ saved: {FIG_DIR}/reconstruction.png")

loaded 339,556 points
✅ saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures/reconstruction.png


In [2]:
# Cell — Generate two rotation GIFs (skeleton + reconstruction)
import os, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

OUT_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/mast3r_output"
FIG_DIR  = "/content/drive/MyDrive/Humanoid_3D_Reconstruction/figures"
os.makedirs(FIG_DIR, exist_ok=True)

N_FRAMES_GIF = 36   # 36 frames = 10° per frame = smooth full turn
FPS          = 12   # plays in 3 seconds


def load_ply(path):
    with open(path) as f:
        skip = 0
        for line in f:
            skip += 1
            if line.strip() == "end_header": break
    return np.loadtxt(path, skiprows=skip)


def make_rotation_gif(path_in, path_out, title,
                       sample=70000, point_size=0.6,
                       split_red_black=False):
    """Render a rotating-camera GIF from a PLY file."""
    data = load_ply(path_in)
    xyz = data[:, :3]; rgb = data[:, 3:6].astype(int)

    if split_red_black:
        # split skeleton ply into cloud / lines / nodes
        is_red   = (rgb[:,0] > 180) & (rgb[:,2] < 80)
        is_black = (rgb[:,0] < 30) & (rgb[:,1] < 30) & (rgb[:,2] < 30)
        line_pts = xyz[is_red]; node_pts = xyz[is_black]
        cloud_pts = xyz[~is_red & ~is_black]
        cloud_rgb = rgb[~is_red & ~is_black] / 255.0
    else:
        cloud_pts, cloud_rgb = xyz, rgb / 255.0
        line_pts = node_pts = np.zeros((0, 3))

    # subsample for speed
    if len(cloud_pts) > sample:
        sel = np.random.choice(len(cloud_pts), sample, replace=False)
        cloud_pts = cloud_pts[sel]; cloud_rgb = cloud_rgb[sel]

    # set up figure once; we just change view_init each frame
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(cloud_pts[:,0], cloud_pts[:,1], cloud_pts[:,2],
               c=cloud_rgb, s=point_size, depthshade=False)
    if len(line_pts):
        ax.scatter(line_pts[:,0], line_pts[:,1], line_pts[:,2],
                   c="#cc1a1a", s=3.5, depthshade=False)
    if len(node_pts):
        ax.scatter(node_pts[:,0], node_pts[:,1], node_pts[:,2],
                   c="k", s=18, depthshade=False)

    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title, fontsize=11)

    def update(i):
        azim = -60 + (360 * i / N_FRAMES_GIF)   # full turn
        ax.view_init(elev=20, azim=azim)
        return []

    print(f"rendering {path_out} ({N_FRAMES_GIF} frames)...")
    anim = FuncAnimation(fig, update, frames=N_FRAMES_GIF, blit=False)
    anim.save(path_out, writer=PillowWriter(fps=FPS))
    plt.close(fig)

    size_mb = os.path.getsize(path_out) / 1e6
    print(f"  ✅ saved: {path_out}  ({size_mb:.1f} MB)")


# 1. Skeleton GIF
make_rotation_gif(
    path_in  = f"{OUT_DIR}/skeleton_final.ply",
    path_out = f"{FIG_DIR}/skeleton_rotation.gif",
    title    = "Structural skeleton — rotating view",
    split_red_black=True,
    point_size=0.7,
)

# 2. Reconstruction GIF
make_rotation_gif(
    path_in  = f"{OUT_DIR}/clean.ply",
    path_out = f"{FIG_DIR}/reconstruction_rotation.gif",
    title    = "Dense reconstruction (MASt3R) — rotating view",
    split_red_black=False,
    point_size=0.6,
)

print(f"\n✅ Both GIFs saved in {FIG_DIR}")

rendering /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures/skeleton_rotation.gif (36 frames)...
  ✅ saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures/skeleton_rotation.gif  (1.2 MB)
rendering /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures/reconstruction_rotation.gif (36 frames)...
  ✅ saved: /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures/reconstruction_rotation.gif  (1.7 MB)

✅ Both GIFs saved in /content/drive/MyDrive/Humanoid_3D_Reconstruction/figures
